In [11]:
import re, json, bisect, collections
from pathlib import Path
import pymupdf
import pandas as pd
from pypdf import PdfReader

HERE = Path.cwd()
RAW = HERE if HERE.name == "ai_lawyer_rawdata" else HERE / "ai_lawyer_rawdata"
PROJECT = RAW.parent / "ai_lawyer"          # generated files live here
INDEX_DIR = PROJECT / "data" / "index"
OUT_DIR = PROJECT / "tests" / "outputs"
for d in (INDEX_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

pdfs = sorted(p for p in RAW.iterdir() if p.suffix.lower() == ".pdf")
print(f"{len(pdfs)} PDFs found in {RAW}")
for p in pdfs: print("  ", p.name)

7 PDFs found in /Users/tusharsarda06/Kuku FM Project/ai_lawyer_rawdata
   Karnataka_Rent_Act_1999.pdf
   Karnataka_Stamp_Act_1957_Tenancy.pdf
   Legal_Services_act_1987.pdf
   Section_17_in_The_Registration_Act_1908.PDF
   THE_INDIAN_LIMITATION_ACT.pdf
   Transfer_of_Property_Act_1882_Sections_105_106_107_108_111_116.pdf
   model-tenancy-act-2021.pdf


In [12]:
MANIFEST = {
    "Karnataka_Rent_Act_1999.pdf": dict(
        code="KRA", act="Karnataka Rent Act, 1999", jurisdiction="Karnataka", status="binding"),

    "Transfer_of_Property_Act_1882_Sections_105_106_107_108_111_116.pdf": dict(
        code="TPA", act="Transfer of Property Act, 1882", jurisdiction="Central",
        status="commentary",         
        format="notes"),

    "Section_17_in_The_Registration_Act_1908.PDF": dict(
        code="REG", act="Registration Act, 1908", jurisdiction="Central", status="binding",
        excerpt=True),

    "Karnataka_Stamp_Act_1957_Tenancy.pdf": dict(
        code="KSA", act="Karnataka Stamp Act, 1957", jurisdiction="Karnataka", status="binding"),

    "THE_INDIAN_LIMITATION_ACT.pdf": dict(
        code="LIM", act="Limitation Act, 1963", jurisdiction="Central", status="binding"),

    "Legal_Services_act_1987.pdf": dict(
        code="LSA", act="Legal Services Authorities Act, 1987", jurisdiction="Central", status="binding"),

    "model-tenancy-act-2021.pdf": dict(
        code="MTA", act="Model Tenancy Act, 2021", jurisdiction="Model law", status="reference_only"),
}

on_disk = {p.name for p in pdfs}
print("In manifest but missing on disk:", sorted(set(MANIFEST) - on_disk))
print("On disk but not in manifest:   ", sorted(on_disk - set(MANIFEST)))
assert len({m["code"] for m in MANIFEST.values()}) == len(MANIFEST), "act codes must be unique"

In manifest but missing on disk: []
On disk but not in manifest:    []


In [13]:
GLYPH_RE = re.compile(r"/G(\d{2,3})")       # pypdf shows undecodable glyphs as /G<ascii code>

def text_quality(text):
    """Share of characters forming real words: clean text ~0.6+, garbled ~0."""
    nonspace = re.sub(r"\s", "", text)
    if not nonspace:
        return 0.0
    words = re.findall(r"[A-Za-z]{3,}", text)
    return sum(len(w) for w in words) / len(nonspace)

def decode_glyphs(text):
    return GLYPH_RE.sub(lambda m: chr(int(m.group(1))), text)

def extract_pages(pdf_path, min_quality=0.4):
    """PyMuPDF first; if a page is garbled, retry with pypdf + glyph decoding."""
    pdf_path = Path(pdf_path)
    pages, fixed_pages, reader = [], [], None
    with pymupdf.open(pdf_path) as doc:
        for i, page in enumerate(doc, start=1):
            text = page.get_text("text")
            if text_quality(text) < min_quality:
                reader = reader or PdfReader(str(pdf_path))
                alt = decode_glyphs(reader.pages[i - 1].extract_text() or "")
                if text_quality(alt) > text_quality(text):
                    text = alt
                fixed_pages.append(i)
            pages.append({"page": i, "text": text})
    return pages, fixed_pages

rows = []
for fname in MANIFEST:
    pages, fixed = extract_pages(RAW / fname)
    scores = [text_quality(p["text"]) for p in pages]
    rows.append({"file": fname, "pages": len(pages),
                 "avg_quality": round(sum(scores) / len(scores), 2),
                 "bad_pages": sum(s < 0.4 for s in scores),
                 "fixed_pages": len(fixed),
                 "first_line": pages[0]["text"].strip().replace("\n", " ")[:70]})
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

,file,pages,avg_quality,bad_pages,fixed_pages,first_line
0,Karnataka_Rent_Act_1999.pdf,77,0.85,0,77,"602 KARNATAKA ACT NO 34 OF 2001 THE KARNATAKA RENT ACT , 1999 Arrangem"
1,Transfer_of_Property_Act_1882_Sections_105_106_107_108_111_116.pdf,8,0.89,0,0,"Study notes • India Page 1 TRANSFER OF PROPERTY ACT, 1882 A section-wi"
2,Section_17_in_The_Registration_Act_1908.PDF,2,0.81,0,0,"Section 17 in The Registration Act, 1908 17. Documents of which regist"
3,Karnataka_Stamp_Act_1957_Tenancy.pdf,106,0.78,0,0,"1 THE KARNATAKA STAMP ACT, 1957 ARRANGEMENT OF SECTIONS Statement"
4,THE_INDIAN_LIMITATION_ACT.pdf,24,0.83,0,0,"1 THE LIMITATION ACT, 1963 _________ ARRANGEMENT OF SECTIONS _____"
5,Legal_Services_act_1987.pdf,19,0.84,0,0,"1 THE LEGAL SERVICES AUTHORITIES ACT, 1987 ________ ARRANGEMENT O"
6,model-tenancy-act-2021.pdf,28,0.83,0,0,Page 1 of 28 ARRANGEMENT OF CLAUSES ________ CHAPTER I PRELIMI


In [14]:
FOOTNOTE_RE = re.compile(
    r"^\s*\d{1,3}\.\s*(Subs\.|Ins\.|Omitted|Added|Rep\.|The words|Vide|Inserted|"
    r"Substituted|Renumbered|Cl\.|Sub-s)", re.I)
AMEND_MARK_RE = re.compile(r"(?<![\w(])\d{1,3}\[")              # India Code markers like "1["
KANOON_FOOTER_RE = re.compile(r"Indian Kanoon - https?://\S+\s*\d*")
PAGE_NUM_RE = re.compile(r"^\s*(Page\s*)?\d{1,4}\s*$", re.I)
CAPS_HEADING_RE = re.compile(r"^\s*(CHAPTER\s+[IVXLC]+[A-Z]?|[A-Z][A-Z ,'&-]{2,70})\s*$")

def clean_pages(pages, repeat_frac=0.5):
    n = len(pages)
    counts = collections.Counter()
    for p in pages:
        counts.update({l.strip() for l in p["text"].splitlines() if l.strip()})
    repeated = {l for l, c in counts.items()                      # running headers/footers
                if n >= 4 and c / n >= repeat_frac and 6 <= len(l) < 120
                and not l.startswith("(") and "SCHEDULE" not in l}
    out = []
    for p in pages:
        keep = []
        for line in p["text"].splitlines():
            s = line.strip()
            if not s or s in repeated or PAGE_NUM_RE.match(s) or FOOTNOTE_RE.match(s):
                continue
            if CAPS_HEADING_RE.match(s) and "SCHEDULE" not in s:
                continue
            keep.append(KANOON_FOOTER_RE.sub("", AMEND_MARK_RE.sub("", line)))
        out.append({"page": p["page"], "text": "\n".join(keep)})
    return out

def join_pages(pages):
    parts, offsets, page_nos, pos = [], [], [], 0
    for p in pages:
        offsets.append(pos); page_nos.append(p["page"])
        parts.append(p["text"]); pos += len(p["text"]) + 1
    return "\n".join(parts), offsets, page_nos

def page_at(offsets, page_nos, char_pos):
    return page_nos[max(bisect.bisect_right(offsets, char_pos) - 1, 0)]

In [24]:
SEC_START_RE = re.compile(r"(?m)^[ \t]*(\d{1,3})([A-Z]{0,3})\.[ \t]*(?=[\[A-Z\"'‘“])")

ORD = "FIRST|SECOND|THIRD|FOURTH|FIFTH|SIXTH|SEVENTH|EIGHTH|NINTH|TENTH"
ORD_NUM = {w: i + 1 for i, w in enumerate(ORD.split("|"))}
SCHED_RE = re.compile(rf"(?m)^[ \t]*(?:THE[ \t]+)?(?:({ORD})[ \t]+)?SCHEDULE\b[^\n]*")
SEE_RE = re.compile(r"^\s*[(\[]\s*See[^)\]]*[)\]]", re.I)     # "(See section 47 and 48)"

def find_body(text):
    """Skip title page and 'Arrangement of Sections' table."""
    m = re.search(r"be it enacted", text, re.I)
    if m:
        return m.start()
    hits = [m.start() for m in re.finditer(r"(?m)^\s*1\.\s+(Short title|Title)", text)]
    return hits[-1] if hits else 0

def select_sections(cands, max_gap=12, allow_any_start=False):
    """Longest increasing chain of section numbers; rejects footnotes and
    numbered list items. cands = [(char_offset, number, suffix)]"""
    n = len(cands)
    NEG = -10**9
    best, prev = [NEG] * n, [-1] * n
    for j in range(n):
        kj = (cands[j][1], cands[j][2])
        if allow_any_start or cands[j][1] <= max_gap:
            best[j] = 1
        for i in range(j):
            if best[i] == NEG:
                continue
            ki = (cands[i][1], cands[i][2])
            if ki < kj and cands[j][1] - cands[i][1] <= max_gap and best[i] + 1 > best[j]:
                best[j], prev[j] = best[i] + 1, i
    if not n or max(best) == NEG:
        return []
    j = max(range(n), key=lambda x: best[x])
    chain = []
    while j != -1:
        chain.append(cands[j]); j = prev[j]
    return chain[::-1]

def section_title(seg):
    head = re.sub(r"\s+", " ", seg[:350])
    head = re.sub(r"^\s*\d{1,3}[A-Z]{0,3}\.\s*", "", head)
    m = re.search(r"^(.{3,200}?)\.?\s*(?:[—–]+|-{1,2})\s*(?=[A-Z(\"'‘“])", head)
    if not m:
        m = re.search(r"^(.{3,120}?)[.:]", head)
    return (m.group(1) if m else head[:80]).strip(" .[]:")

def split_words(text, max_words=300, overlap=50):
    words = text.split()
    if len(words) <= max_words:
        return [" ".join(words)]
    out, step = [], max_words - overlap
    for s in range(0, len(words), step):
        out.append(" ".join(words[s:s + max_words]))
        if s + max_words >= len(words):
            break
    return out

In [25]:
def _base(pdf_path, meta):
    return {"act_code": meta["code"], "act": meta["act"],
            "jurisdiction": meta["jurisdiction"], "status": meta["status"],
            "source_file": Path(pdf_path).name, "source_url": meta.get("source", "")}

def _next_section_start(cands, off, num, default):
    """Where the next higher-numbered section begins (for keep_sections ranges)."""
    later = [c[0] for c in cands if c[0] > off and c[1] > num]
    return later[0] if later else default

def schedule_chunks(text, start, base, meta, max_words=300, overlap=50):
    """Split the schedule area at each 'FIFTH SCHEDULE'-style heading and label it,
    e.g. 'Fifth Schedule (See section 47 and 48)'."""
    heads = list(SCHED_RE.finditer(text, start))
    out = []
    for i, h in enumerate(heads):
        end = heads[i + 1].start() if i + 1 < len(heads) else len(text)
        num = ORD_NUM.get(h.group(1), 1 if len(heads) == 1 else i + 1)
        see = SEE_RE.match(text[h.end():h.end() + 120])
        title = h.group(0).strip().title() + (" " + see.group(0).strip() if see else "")
        parts = split_words(text[h.start():end], max_words, overlap)
        for k, part in enumerate(parts, start=1):
            out.append({**base, "chunk_id": f"{meta['code']}-sch{num}" + (f"-p{k}" if len(parts) > 1 else ""),
                        "unit": "schedule", "section": f"Schedule {num}", "title": title,
                        "part": k, "n_parts": len(parts), "page": None, "text": part,
                        "embed_text": f"{meta['act']} | {title}\n{part}"})
    return out

def chunk_act(pdf_path, meta, max_words=300, overlap=50, max_gap=12):
    """Bare Act text: sections like '27. Protection of tenants...'"""
    pages, fixed = extract_pages(pdf_path)
    text, offsets, page_nos = join_pages(clean_pages(pages))

    body_start = find_body(text)
    sm = None if meta.get("no_schedule") else SCHED_RE.search(text, body_start)
    body_end = sm.start() if sm else len(text)

    cands = [(m.start(), int(m.group(1)), m.group(2))
             for m in SEC_START_RE.finditer(text, body_start, body_end)]
    secs = select_sections(cands, max_gap=max_gap,
                           allow_any_start=meta.get("excerpt", False))

    keep = meta.get("keep_sections")
    if keep:
        lo, hi = keep
        secs = [s for s in secs if lo <= s[1] <= hi]
        sm = None

    base, chunks = _base(pdf_path, meta), []
    for idx, (off, num, suf) in enumerate(secs):
        if idx + 1 < len(secs):
            end = secs[idx + 1][0]
        elif keep:
            end = _next_section_start(cands, off, num, body_end)
        else:
            end = body_end
        seg = text[off:end].strip()
        sec, title = f"{num}{suf}", section_title(seg)
        parts = split_words(seg, max_words, overlap)
        for k, part in enumerate(parts, start=1):
            cid = f"{meta['code']}-s{sec}" + (f"-p{k}" if len(parts) > 1 else "")
            chunks.append({**base, "chunk_id": cid, "unit": "section", "section": sec,
                           "title": title, "part": k, "n_parts": len(parts),
                           "page": page_at(offsets, page_nos, off), "text": part,
                           "embed_text": f"{meta['act']} | Section {sec}: {title}\n{part}"})

    if sm:
        chunks += schedule_chunks(text, sm.start(), base, meta, max_words, overlap)

    nums = sorted({n for _, n, _ in secs})
    n_sched = len({c["section"] for c in chunks if c["unit"] == "schedule"})
    diag = {"file": Path(pdf_path).name, "code": meta["code"], "pages": len(pages),
            "fixed_pages": len(fixed), "sections_found": len(secs),
            "range": f"{nums[0]}–{nums[-1]}" if nums else "none",
            "missing_numbers": [n for n in range(nums[0], nums[-1] + 1) if n not in set(nums)] if nums else [],
            "schedules": n_sched, "chunks": len(chunks)}
    return chunks, diag


NOTE_SEC_RE = re.compile(r"(?m)^[ \t]*Section[ \t]+(\d{1,3}[A-Z]?)[ \t]*[—–-][ \t]*(.+?)[ \t]*$")
NOTE_END_RE = re.compile(r"(?m)^[ \t]*(Quick revision chart|Sources and statutory reference)")

def chunk_notes(pdf_path, meta, max_words=300, overlap=50):
    """Study-notes PDFs with headings like 'Section 106 — Title'."""
    pages, fixed = extract_pages(pdf_path)
    text, offsets, page_nos = join_pages(clean_pages(pages))
    hits = list(NOTE_SEC_RE.finditer(text))
    endm = NOTE_END_RE.search(text, hits[-1].end()) if hits else None
    doc_end = endm.start() if endm else len(text)

    base, chunks = _base(pdf_path, meta), []
    for i, m in enumerate(hits):
        end = hits[i + 1].start() if i + 1 < len(hits) else doc_end
        sec, title = m.group(1), m.group(2).strip()
        parts = split_words(text[m.start():end], max_words, overlap)
        for k, part in enumerate(parts, start=1):
            cid = f"{meta['code']}-s{sec}" + (f"-p{k}" if len(parts) > 1 else "")
            chunks.append({**base, "chunk_id": cid, "unit": "summary", "section": sec,
                           "title": title, "part": k, "n_parts": len(parts),
                           "page": page_at(offsets, page_nos, m.start()), "text": part,
                           "embed_text": f"{meta['act']} | Section {sec} (summary): {title}\n{part}"})

    nums = sorted({int(re.match(r"\d+", m.group(1)).group()) for m in hits})
    diag = {"file": Path(pdf_path).name, "code": meta["code"], "pages": len(pages),
            "fixed_pages": len(fixed), "sections_found": len(hits),
            "range": f"{nums[0]}–{nums[-1]}" if nums else "none",
            "missing_numbers": "n/a (selected sections)", "schedules": 0,
            "chunks": len(chunks)}
    return chunks, diag

In [26]:
all_chunks, diags = [], []
for fname, meta in MANIFEST.items():
    parser = chunk_notes if meta.get("format") == "notes" else chunk_act
    chunks, diag = parser(RAW / fname, meta)
    all_chunks += chunks
    diags.append(diag)

ids = [c["chunk_id"] for c in all_chunks]
dupes = [i for i, n in collections.Counter(ids).items() if n > 1]
print(f"{len(all_chunks)} chunks total | duplicate ids: {dupes[:10]}")

def peek(fname, page=1, n=1200):
    """Raw text of one PDF page (after decode fix)."""
    pages, _ = extract_pages(RAW / fname)
    print(f"===== {fname} | page {page} of {len(pages)} =====")
    print(pages[page - 1]["text"][:n])

def show(code, n=5, start=0):
    """Print detected sections of one Act."""
    for c in [c for c in all_chunks if c["act_code"] == code][start:start + n]:
        print(f"[{c['chunk_id']}] p.{c['page']} — {c['title']}\n{c['text'][:300]}\n{'-'*90}")

diag_df = pd.DataFrame(diags)
diag_df.to_csv(OUT_DIR / "ingest_diagnostics.csv", index=False)
diag_df

458 chunks total | duplicate ids: []


,file,code,pages,fixed_pages,sections_found,range,missing_numbers,schedules,chunks
0,Karnataka_Rent_Act_1999.pdf,KRA,77,77,70,1–70,[],3,107
1,Transfer_of_Property_Act_1882_Sections_105_106_107_108_111_116.pdf,TPA,8,0,6,105–116,n/a (selected sections),0,10
2,Section_17_in_The_Registration_Act_1908.PDF,REG,2,0,1,17–17,[],0,4
3,Karnataka_Stamp_Act_1957_Tenancy.pdf,KSA,106,0,79,1–72,"[5, 6, 7, 8, 9]",1,188
4,THE_INDIAN_LIMITATION_ACT.pdf,LIM,24,0,32,1–32,[],1,56
5,Legal_Services_act_1987.pdf,LSA,19,0,40,1–30,[],0,51
6,model-tenancy-act-2021.pdf,MTA,28,0,19,2–45,"[3, 4, 5, 7, 9, 11, 13, 14, 15, 17, 19, 20, 21, 22, 24, 26, 27, 32, 35, 36, ...",2,42


In [27]:
def show_sections(code, sections, chars=400):
    """Print specific sections by number, e.g. show_sections("KRA", ["47", "48", "49"])."""
    for c in all_chunks:
        if c["act_code"] == code and c["section"] in sections and c["part"] == 1:
            print(f"[{c['chunk_id']}] p.{c['page']} — {c['title']} ({c['n_parts']} part(s))")
            print(c["text"][:chars])
            print("-" * 90)

show_sections("KRA", ["47", "48", "49"])

[KRA-s47-p1] p.57 — Landlords duty to keep the premises in good repair (2 part(s))
47. Landlords duty to keep the premises in good repair.- (1) Subject to any contract in writing to the contrary every landlord shall be bound to keep the premises in good and tenantable repairs in relation to matters falling under Part A of Schedule V. Explanation.- Good and tenantable repairs under this section and under section 49 shall mean such repairs as shall keep the premises in the same c
------------------------------------------------------------------------------------------
[KRA-s48-p1] p.58 — Duties of tenant (2 part(s))
48. Duties of tenant.- (1) Every tenant shall be bound to keep the premises in good and tenantable repairs in relation to matters falling under Part B of Schedule V. (2) Where any repairs, in relation to a matter falling under Part B of Schedule V, without which the premises are not habitable or useable except with undue inconvenience are to be made and the tenant neglects

In [28]:
import numpy as np
import faiss, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "intfloat/multilingual-e5-base"   # English + Hindi + Kannada
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
model = SentenceTransformer(EMBED_MODEL, device=device)
print("device:", device)

def tokenize(text):
    return re.findall(r"\w+", text.lower())

def build_index(chunks, model, index_dir, batch_size=32):
    emb = model.encode(["passage: " + c["embed_text"] for c in chunks],   # e5 needs "passage: "
                       batch_size=batch_size, normalize_embeddings=True,
                       show_progress_bar=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])      # cosine similarity on unit vectors
    index.add(emb)
    faiss.write_index(index, str(index_dir / "dense.faiss"))
    with open(index_dir / "chunks.jsonl", "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    json.dump({"embed_model": EMBED_MODEL, "n_chunks": len(chunks), "dim": int(emb.shape[1])},
              open(index_dir / "config.json", "w"))
    return index

build_index(all_chunks, model, INDEX_DIR)
print(f"Indexed {len(all_chunks)} chunks → {INDEX_DIR}")

device: mps


Batches: 100%|█████████████████████████████████| 15/15 [00:14<00:00,  1.07it/s]


Indexed 458 chunks → /Users/tusharsarda06/Kuku FM Project/ai_lawyer/data/index


In [29]:
class LawRetriever:
    def __init__(self, index_dir, model):
        self.chunks = [json.loads(l) for l in open(index_dir / "chunks.jsonl", encoding="utf-8")]
        self.by_id = {c["chunk_id"]: c for c in self.chunks}
        self.model = model
        self.index = faiss.read_index(str(index_dir / "dense.faiss"))
        self.bm25 = BM25Okapi([tokenize(c["embed_text"]) for c in self.chunks])

    def search(self, query, k=6, pool=50, acts=None, include_reference=False, rrf_k=60):
        """Hybrid search (meaning + keywords). Returns chunks with:
           dense = cosine similarity (used later for the 'I don't know' threshold)
           rrf   = fused rank score"""
        q = self.model.encode(["query: " + query], normalize_embeddings=True).astype("float32")
        _, I = self.index.search(q, pool)
        dense_ids = [int(i) for i in I[0] if i >= 0]
        bm = self.bm25.get_scores(tokenize(query))
        sparse_ids = [int(i) for i in np.argsort(-bm)[:pool] if bm[i] > 0]

        fused = collections.defaultdict(float)          # reciprocal rank fusion
        for r, i in enumerate(dense_ids):
            fused[i] += 1 / (rrf_k + r + 1)
        for r, i in enumerate(sparse_ids):
            fused[i] += 1 / (rrf_k + r + 1)

        results = []
        for i, s in sorted(fused.items(), key=lambda x: -x[1]):
            c = self.chunks[i]
            if acts and c["act_code"] not in acts:
                continue
            if not include_reference and c["status"] == "reference_only":   # keeps MTA out
                continue
            cos = float(q[0] @ self.index.reconstruct(i))
            results.append({**c, "rrf": round(s, 5), "dense": round(cos, 4)})
            if len(results) == k:
                break
        return results

    def get(self, chunk_id):
        """Exact lookup; the citation verifier uses this later."""
        return self.by_id.get(chunk_id)

retriever = LawRetriever(INDEX_DIR, model)
for r in retriever.search("landlord cut off my water supply", k=3):
    print(f"{r['chunk_id']:<14} dense={r['dense']:.3f}  {r['title'][:60]}")

KRA-s49-p1     dense=0.862  Cutting off or withholding essential supply or service
KRA-s49-p2     dense=0.859  Cutting off or withholding essential supply or service
KRA-s54-p4     dense=0.832  Offences and penalties


In [30]:
# (query, acceptable act codes, acceptable sections or None = any section of those acts)
TESTS = [
    ("Landlord cut off water and electricity to force me out",            {"KRA"}, {"49"}),
    ("My landlord is not giving me receipts for rent I paid",             {"KRA"}, {"16"}),
    ("Landlord refuses to accept my rent, what can I do?",                {"KRA"}, {"17"}),
    ("On what grounds can my landlord evict me?",                         {"KRA"}, {"27"}),
    ("Who has to do repairs, the landlord or the tenant?",                {"KRA", "TPA"}, {"47", "48", "108", "Schedule 5"}),
    ("Who pays for fixing a leaking tap?",                                {"KRA"}, {"Schedule 5"}),
    ("How much notice is needed to end a month-to-month tenancy?",        {"TPA"}, {"106"}),
    ("Does an 11 month rent agreement have to be registered?",            {"REG", "TPA"}, {"17", "107"}),
    ("I stayed after the lease ended and landlord took rent, am I still a tenant?", {"TPA"}, {"116"}),
    ("How much stamp duty on a rental agreement?",                        {"KSA"}, None),
    ("How long do I have to file a case to recover money owed to me?",    {"LIM"}, None),
    ("Can I get a free lawyer from the government?",                      {"LSA"}, None),
    ("मकान मालिक ने पानी और बिजली काट दी है",                               {"KRA"}, {"49"}),
    ("ಮನೆ ಮಾಲೀಕರು ನನ್ನನ್ನು ಮನೆಯಿಂದ ಹೊರಹಾಕಲು ಬಯಸುತ್ತಾರೆ",                    {"KRA"}, {"27"}),
]

rows = []
for q, acts, secs in TESTS:
    res = retriever.search(q, k=5)
    def ok(r): return r["act_code"] in acts and (secs is None or r["section"] in secs)
    rank = next((i + 1 for i, r in enumerate(res) if ok(r)), None)
    rows.append({"query": q[:55], "hit@5": rank is not None, "rank": rank,
                 "top_dense": res[0]["dense"] if res else None,
                 "top5": ", ".join(r["chunk_id"] for r in res)})
eval_df = pd.DataFrame(rows)
print(f"hit@5 = {eval_df['hit@5'].mean():.0%}   |   hit@1 = {(eval_df['rank'] == 1).mean():.0%}")
eval_df

hit@5 = 100%   |   hit@1 = 93%


,query,hit@5,rank,top_dense,top5
0,Landlord cut off water and electricity to force me out,True,1,0.8448,"KRA-s49-p1, KRA-s49-p2, KRA-s54-p4, TPA-s108-p1, KRA-s8"
1,My landlord is not giving me receipts for rent I paid,True,1,0.8466,"KRA-s16-p1, KRA-s16-p2, KRA-s49-p1, KRA-s17-p3, KRA-s27-p2"
2,"Landlord refuses to accept my rent, what can I do?",True,2,0.8405,"KRA-s16-p2, KRA-s17-p3, KRA-s16-p1, KRA-s17-p1, KRA-s43"
3,On what grounds can my landlord evict me?,True,1,0.8476,"KRA-s27-p1, TPA-s108-p1, KRA-s42-p2, KRA-s27-p7, KRA-s27-p9"
4,"Who has to do repairs, the landlord or the tenant?",True,1,0.8539,"KRA-s48-p1, KRA-s47-p1, TPA-s108-p3, KRA-s36-p1, KRA-s27-p3"
5,Who pays for fixing a leaking tap?,True,1,0.7944,"KRA-sch5, KRA-s12-p2, KRA-s27-p6, KRA-s12-p1, TPA-s108-p1"
6,How much notice is needed to end a month-to-month tenan,True,1,0.8448,"TPA-s106-p1, TPA-s106-p2, KRA-s33, TPA-s116, KRA-s27-p2"
7,Does an 11 month rent agreement have to be registered?,True,1,0.8460,"TPA-s107, KRA-s2-p1, KRA-s54-p1, KRA-s27-p2, KRA-s11"
8,"I stayed after the lease ended and landlord took rent,",True,1,0.8525,"TPA-s116, TPA-s108-p3, TPA-s108-p1, KRA-s43, KRA-s37-p1"
9,How much stamp duty on a rental agreement?,True,1,0.8467,"KSA-s25-p2, KSA-sch1-p29, KSA-s25-p1, TPA-s107, KSA-sch1-p5"


In [31]:
TRAPS = [
    "How do I file for divorce?",                                         # out of scope
    "best biryani in Bangalore",                                          # nonsense
    "My employer has not paid my salary for 3 months",                    # different legal area
    "What does Section 999 of the Karnataka Rent Act say?",               # non-existent section
    "What is the 2-month security deposit cap in the Karnataka Rent Act?",# common myth
]
for q in TRAPS:
    res = retriever.search(q, k=3)
    print(f"{res[0]['dense']:.3f}  {q[:60]:<60} -> {[r['chunk_id'] for r in res]}")

print("\nIn-scope top_dense: min {:.3f} | median {:.3f}".format(
    eval_df["top_dense"].min(), eval_df["top_dense"].median()))

0.797  How do I file for divorce?                                   -> ['KRA-s21', 'LSA-s12', 'KRA-s42-p3']
0.773  best biryani in Bangalore                                    -> ['KSA-sch1-p27', 'KSA-sch1-p18', 'KSA-sch1-p45']
0.792  My employer has not paid my salary for 3 months              -> ['KSA-s52', 'LIM-sch1-p6', 'LIM-sch1-p2']
0.873  What does Section 999 of the Karnataka Rent Act say?         -> ['KRA-s3-p3', 'KRA-s65', 'KRA-s1']
0.858  What is the 2-month security deposit cap in the Karnataka Re -> ['KRA-s18', 'KRA-s27-p2', 'KRA-s54-p2']

In-scope top_dense: min 0.763 | median 0.845


In [23]:
eval_df.to_csv(OUT_DIR / "retrieval_eval.csv", index=False)
print("saved:", OUT_DIR / "retrieval_eval.csv", "and", OUT_DIR / "ingest_diagnostics.csv")

saved: /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/retrieval_eval.csv and /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/ingest_diagnostics.csv


In [32]:
show_sections("KRA", ["Schedule 5"])

[KRA-sch5] p.None — Fifth Schedule (See section 47 and 48) (1 part(s))
FIFTH SCHEDULE (See section 47 and 48) A. Structural repairs to be got done by the landlord. 1. Structural repairs, except those necessitated by damage caused by the tenant. 2. Whitewashing of walls and painting of doors and windows once in three years. 3. Changing and plumbing pipes when necessary. 4. Internal and external wiring and related maintenance . B. Day to day repairs to be got done by t
------------------------------------------------------------------------------------------


In [33]:
ACT_ALIASES = {
    "KRA": ["rent act", "karnataka rent"],
    "TPA": ["transfer of property", "tpa"],
    "REG": ["registration act"],
    "KSA": ["stamp act"],
    "LIM": ["limitation act"],
    "LSA": ["legal services"],
}
SECTION_REF_RE = re.compile(r"\b(?:section|sec\.?|s\.)\s*(\d{1,3}[A-Z]?)\b", re.I)

KNOWN_SECTIONS = collections.defaultdict(set)
for c in retriever.chunks:
    KNOWN_SECTIONS[c["act_code"]].add(c["section"])

def check_section_refs(text):
    """Find 'Section N' mentions and check whether that section is in the corpus.
    'not in corpus' ≠ 'does not exist': for TPA only selected sections are indexed."""
    low = text.lower()
    acts = [code for code, names in ACT_ALIASES.items() if any(n in low for n in names)]
    results = []
    for m in SECTION_REF_RE.finditer(text):
        sec = m.group(1).upper()
        targets = acts or list(ACT_ALIASES)
        found = [a for a in targets if sec in KNOWN_SECTIONS[a]]
        results.append({"section": sec, "acts_checked": targets, "in_corpus": found})
    return results

for q in ["What does Section 999 of the Karnataka Rent Act say?",
          "Explain section 49 of the rent act",
          "What does section 106 of the Transfer of Property Act say?",
          "What is section 110 of TPA?",
          "Is there anything in sec 27 about eviction?"]:
    print(f"{q[:58]:<58} -> {check_section_refs(q)}")

What does Section 999 of the Karnataka Rent Act say?       -> [{'section': '999', 'acts_checked': ['KRA'], 'in_corpus': []}]
Explain section 49 of the rent act                         -> [{'section': '49', 'acts_checked': ['KRA'], 'in_corpus': ['KRA']}]
What does section 106 of the Transfer of Property Act say? -> [{'section': '106', 'acts_checked': ['TPA'], 'in_corpus': ['TPA']}]
What is section 110 of TPA?                                -> [{'section': '110', 'acts_checked': ['TPA'], 'in_corpus': []}]
Is there anything in sec 27 about eviction?                -> [{'section': '27', 'acts_checked': ['KRA', 'TPA', 'REG', 'KSA', 'LIM', 'LSA'], 'in_corpus': ['KRA', 'KSA', 'LIM', 'LSA']}]


In [34]:
eval_df.to_csv(OUT_DIR / "retrieval_eval.csv", index=False)
print("saved:", OUT_DIR / "retrieval_eval.csv", "and", OUT_DIR / "ingest_diagnostics.csv")

saved: /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/retrieval_eval.csv and /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/ingest_diagnostics.csv


In [35]:
GROQ_API_KEY = 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

NameError: name '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' is not defined

In [38]:
import getpass
env_file = PROJECT / ".env"
key = getpass.getpass("Paste your Groq API key (hidden): ").strip()
env_file.write_text(f"GROQ_API_KEY={key}\n")
env_file.chmod(0o600)                        # only your user can read it
print("Saved to", env_file)

Paste your Groq API key (hidden):  ········


Saved to /Users/tusharsarda06/Kuku FM Project/ai_lawyer/.env


In [39]:
import os, time, getpass
from groq import Groq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
client = Groq()

available = sorted(m.id for m in client.models.list().data)
print("Live Groq models:", available)

def pick(preferences):
    for p in preferences:
        if p in available:
            return p
    raise ValueError(f"None of {preferences} available — choose from the list above")

MODEL_BIG   = pick(["openai/gpt-oss-120b", "llama-3.3-70b-versatile"])               # answers
MODEL_SMALL = pick(["openai/gpt-oss-20b", "llama-3.1-8b-instant", MODEL_BIG])        # intake
print("MODEL_BIG =", MODEL_BIG, "| MODEL_SMALL =", MODEL_SMALL)

# USD per 1M tokens (input, output) — Groq list prices, Aug 2026. Free tier costs nothing,
# but we log what it WOULD cost at paid rates for the "cost at real-world volume" section.
PRICES = {"openai/gpt-oss-120b": (0.15, 0.60), "openai/gpt-oss-20b": (0.075, 0.30)}
USAGE_LOG = []

def parse_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        raise ValueError(f"No JSON in model output: {text[:200]}")
    return json.loads(m.group(0))

def llm(messages, model, json_mode=False, temperature=0.2, max_tokens=3000, retries=3, tag=""):
    kwargs = dict(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    for attempt in range(retries):
        try:
            t0 = time.time()
            r = client.chat.completions.create(**kwargs)
            break
        except Exception as e:                       # rate limits on free tier → back off
            if attempt == retries - 1:
                raise
            wait = 4 * (2 ** attempt)
            print(f"  LLM error ({type(e).__name__}), retrying in {wait}s")
            time.sleep(wait)
    u = r.usage
    pin, pout = PRICES.get(model, (0, 0))
    USAGE_LOG.append({"tag": tag, "model": model, "in_tok": u.prompt_tokens, "out_tok": u.completion_tokens,
                      "secs": round(time.time() - t0, 2),
                      "usd": (u.prompt_tokens * pin + u.completion_tokens * pout) / 1e6})
    text = r.choices[0].message.content or ""
    return parse_json(text) if json_mode else text

print(llm([{"role": "user", "content": "Reply with just the word OK."}], MODEL_SMALL, tag="ping"))

Groq API key:  ········


Live Groq models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']
MODEL_BIG = openai/gpt-oss-120b | MODEL_SMALL = openai/gpt-oss-20b
OK


In [40]:
TOPICS = ["deposit_refund", "eviction_threat", "essential_services_cut", "repairs",
          "rent_increase", "rent_payment_or_receipts", "agreement_registration_stamp",
          "notice_to_vacate", "holding_over", "legal_aid", "other_tenancy", "out_of_scope"]
FLAGS = ["violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice",
         "deadline_within_7_days", "commercial_premises", "outside_karnataka"]

INTAKE_SYSTEM = f"""You are the intake step of a help tool for RESIDENTIAL tenancy problems in Bengaluru / Karnataka, India.
Read the user's message and extract facts. Do NOT give legal advice. Never invent dates, amounts or facts:
use null or empty lists when something is not stated.
Return ONLY a JSON object with these keys:
- "language": "en", "hi", "kn", or "other" (the language the user wrote in; Hinglish counts as "hi")
- "topic": one of {TOPICS}
- "in_scope": true only if this is a residential landlord-tenant matter (Karnataka or location unstated)
- "role": "tenant", "landlord" or "unknown"
- "english_summary": 1-3 neutral English sentences describing the problem and what the user wants
- "facts": {{"timeline": [{{"date": str|null, "event": str}}], "amounts_inr": [{{"what": str, "amount": number}}],
             "evidence": [str], "agreement": "written registered" | "written unregistered" | "oral" | null}}
- "missing_info": up to 3 short questions whose answers would change the advice
- "urgent_flags": subset of {FLAGS}"""

# Code-side red flags: these never depend on the model noticing them
RED_FLAG_RULES = {
    "violence_or_threat": r"\b(threat\w*|beat\w*|hit me|assault\w*|violen\w*|goons?|dhamki|maar\w*)\b",
    "locked_out_or_belongings_removed": r"\b(locked (me )?out|changed the locks?|threw (out )?my|removed my (things|belongings))\b",
    "court_or_police_notice": r"\b(summons|court notice|fir|police|legal notice|lawyer'?s? notice|vakil)\b",
}
LAWYER_FLAGS = {"violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice"}
LARGE_AMOUNT_INR = 500_000

def intake(user_text):
    data = llm([{"role": "system", "content": INTAKE_SYSTEM},
                {"role": "user", "content": user_text}], MODEL_SMALL, json_mode=True, tag="intake")
    flags = set(data.get("urgent_flags") or [])
    for flag, pattern in RED_FLAG_RULES.items():
        if re.search(pattern, user_text, re.I):
            flags.add(flag)
    data["urgent_flags"] = sorted(flags)
    data["section_refs"] = check_section_refs(user_text)          # Cell 14

    amounts = [a.get("amount") or 0 for a in (data.get("facts") or {}).get("amounts_inr") or []]
    reasons = [f for f in data["urgent_flags"] if f in LAWYER_FLAGS]
    if amounts and max(amounts) >= LARGE_AMOUNT_INR:
        reasons.append(f"amount at stake ≥ ₹{LARGE_AMOUNT_INR:,}")
    data["escalate"] = {"needs_lawyer": bool(reasons), "reasons": reasons}
    return data

demo = intake("Bhai mera landlord 40,000 ka deposit wapas nahi de raha. Maine 10 March ko flat khali kiya tha, "
              "agreement 11 mahine ka tha. Ab phone bhi nahi utha raha.")
print(json.dumps(demo, indent=2, ensure_ascii=False))

{
  "language": "hi",
  "topic": "deposit_refund",
  "in_scope": true,
  "role": "tenant",
  "english_summary": "The tenant vacated the flat on 10 March and is seeking the return of a ₹40,000 deposit that the landlord has not refunded. The landlord is unresponsive to phone calls.",
  "facts": {
    "timeline": [
      {
        "date": "2023-03-10",
        "event": "Tenant vacated the flat"
      }
    ],
    "amounts_inr": [
      {
        "what": "Deposit",
        "amount": 40000
      }
    ],
    "evidence": [
      "Landlord not answering phone"
    ],
    "agreement": null
  },
  "missing_info": [
    "Did you give the landlord written notice before vacating?",
    "Did you return the keys and hand over the premises in good condition?",
    "Have you documented any communication with the landlord regarding the deposit?"
  ],
  "urgent_flags": [],
  "section_refs": [],
  "escalate": {
    "needs_lawyer": false,
    "reasons": []
  }
}


In [64]:
#Updated Cell 17 
TOPICS = ["deposit_refund", "eviction_threat", "essential_services_cut", "repairs",
          "rent_increase", "rent_payment_or_receipts", "agreement_registration_stamp",
          "notice_to_vacate", "holding_over", "legal_aid", "other_tenancy", "out_of_scope"]
FLAGS = ["violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice",
         "deadline_within_7_days", "commercial_premises", "outside_karnataka"]

INTAKE_SYSTEM = f"""You are the intake step of a help tool for RESIDENTIAL tenancy problems in Bengaluru / Karnataka, India.
Read the user's message and extract facts. Do NOT give legal advice. Never invent dates, years, amounts or facts:
use null or empty lists when something is not stated. Write dates exactly as precise as the user gave them.
Return ONLY a JSON object with these keys:
- "language": "en", "hi" (Hindi in Devanagari), "hinglish" (Hindi in Roman letters), "kn", or "other"
- "topic": one of {TOPICS}
- "in_scope": true only if this is a residential landlord-tenant matter (Karnataka or location unstated)
- "role": "tenant", "landlord" or "unknown"
- "english_summary": 1-3 neutral English sentences describing the problem and what the user wants
- "facts": {{"timeline": [{{"date": str|null, "event": str}}], "amounts_inr": [{{"what": str, "amount": number}}],
             "evidence": [str], "agreement": "written registered" | "written unregistered" | "oral" | null}}
- "missing_info": up to 3 short questions whose answers would change the advice
- "urgent_flags": subset of {FLAGS}"""

# Code-side red flags: these never depend on the model noticing them
RED_FLAG_RULES = {
    "violence_or_threat": r"\b(threat\w*|beat\w*|hit me|assault\w*|violen\w*|goons?|dhamki|maar\w*)\b",
    "locked_out_or_belongings_removed": r"\b(locked (me )?out|changed the locks?|threw (out )?my|removed my (things|belongings))\b",
    "court_or_police_notice": r"\b(summons|court notice|fir|police|legal notice|lawyer'?s? notice|vakil)\b",
}
LAWYER_FLAGS = {"violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice"}
LARGE_AMOUNT_INR = 500_000

def fix_invented_years(case, user_text):
    """Code check: if the model added a year the user never wrote, remove it."""
    for ev in (case.get("facts") or {}).get("timeline") or []:
        d = ev.get("date") or ""
        m = re.match(r"((?:19|20)\d{2})-(\d{2})-(\d{2})$", d)
        if m and m.group(1) not in user_text:
            ev["date"] = datetime.strptime(d, "%Y-%m-%d").strftime("%d %B") + " (year not stated)"

def guess_language(text):
    """Script-based guess, used only when the intake model fails."""
    if re.search(r"[\u0C80-\u0CFF]", text): return "kn"
    if re.search(r"[\u0900-\u097F]", text): return "hi"
    return "en"

def default_case(user_text):
    """Safe defaults if the intake model crashes or returns bad JSON."""
    return {"language": guess_language(user_text), "topic": "other_tenancy", "in_scope": True, "role": "unknown",
            "english_summary": user_text, "facts": {"timeline": [], "amounts_inr": [], "evidence": [], "agreement": None},
            "missing_info": [], "urgent_flags": [], "intake_failed": True}

def intake(user_text):
    try:
        data = llm([{"role": "system", "content": INTAKE_SYSTEM},
                    {"role": "user", "content": user_text}], MODEL_SMALL, json_mode=True, tag="intake")
        if not isinstance(data, dict) or "topic" not in data:
            raise ValueError("intake JSON missing keys")
    except Exception as e:
        print(f"  intake failed ({type(e).__name__}) → safe defaults")
        data = default_case(user_text)
    fix_invented_years(data, user_text)

    flags = set(data.get("urgent_flags") or [])
    for flag, pattern in RED_FLAG_RULES.items():
        if re.search(pattern, user_text, re.I):
            flags.add(flag)
    data["urgent_flags"] = sorted(flags)
    data["section_refs"] = check_section_refs(user_text)          # Cell 14

    amounts = [a.get("amount") or 0 for a in (data.get("facts") or {}).get("amounts_inr") or []]
    reasons = [f for f in data["urgent_flags"] if f in LAWYER_FLAGS]
    if amounts and max(amounts) >= LARGE_AMOUNT_INR:
        reasons.append(f"amount at stake ≥ ₹{LARGE_AMOUNT_INR:,}")
    data["escalate"] = {"needs_lawyer": bool(reasons), "reasons": reasons}
    return data

print("intake() ready")

intake() ready


In [41]:
LANG_NAMES = {"en": "English", "hi": "Hindi (simple, Devanagari; Hinglish is fine if the user wrote Hinglish)",
              "kn": "Kannada", "other": "English"}

ADVISOR_SYSTEM = """You help ordinary people in Bengaluru understand residential tenancy problems.
STRICT RULES:
1. Use ONLY the SOURCES given. Every legal statement must end with its source ID in square brackets, e.g. [KRA-s49-p1].
2. Never mention any Act, section, schedule, rule, court case or judgment that is not in the SOURCES. Do not cite case law at all.
3. If the SOURCES do not answer the question, say so plainly in "what_i_dont_know". Never fill gaps from memory.
4. Sources marked "summary" are plain-language summaries, not the bare Act: say "(summary)" when you rely on them.
5. Remind the user that their rent agreement's terms matter and the Act may not apply to every rental.
6. Plain words, short sentences, no legal jargon. Write the answer in: {language}.
Return ONLY a JSON object:
{{"where_you_stand": str, "options": [str], "next_steps": [str], "what_i_dont_know": str,
  "needs_lawyer": bool, "lawyer_reason": str}}"""

def format_sources(chunks):
    blocks = []
    for c in chunks:
        label = c["section"] if c["section"].startswith("Schedule") else f"Section {c['section']}"
        kind = " (summary, not bare text)" if c["status"] == "commentary" else ""
        blocks.append(f"[{c['chunk_id']}] {c['act']}, {label}{kind}: {c['title']}\n{c['text']}")
    return "\n\n".join(blocks)

def advise(user_text, case, sources, feedback=None):
    msgs = [{"role": "system", "content": ADVISOR_SYSTEM.format(language=LANG_NAMES.get(case.get("language"), "English"))},
            {"role": "user", "content":
                f"USER MESSAGE:\n{user_text}\n\nEXTRACTED FACTS:\n{json.dumps(case.get('facts'), ensure_ascii=False)}\n\n"
                f"SOURCES:\n{format_sources(sources)}"}]
    if feedback:
        msgs.append({"role": "user", "content": f"Your previous answer failed verification: {feedback}\n"
                                                 "Rewrite it following the rules exactly."})
    return llm(msgs, MODEL_BIG, json_mode=True, temperature=0.1, tag="advise")

In [42]:
ID_GROUP_RE = re.compile(r"\[([A-Z]{3}-[^\]]+)\]")
ID_RE = re.compile(r"[A-Z]{3}-(?:s\d{1,3}[A-Z]{0,3}|sch\d+)(?:-p\d+)?")
SEC_MENTION_RE = re.compile(r"\b(?:section|sec\.?|s\.)\s*(\d{1,3}[A-Z]?)\b", re.I)
SCHED_MENTION_RE = re.compile(rf"\b({ORD})\s+schedule\b", re.I)
CASE_LAW_RE = re.compile(r"\b(v\.|vs\.?|versus)\s+[A-Z]|\bAIR\s+\d{{4}}|\(\d{{4}}\)\s*\d+\s*SCC".replace("{{", "{").replace("}}", "}"))

def answer_text(ans):
    return " ".join([ans.get("where_you_stand", ""), *ans.get("options", []),
                     *ans.get("next_steps", []), ans.get("what_i_dont_know", "")])

def verify(ans, sources):
    text = answer_text(ans)
    allowed_ids = {c["chunk_id"] for c in sources}
    allowed_secs = {c["section"] for c in sources}
    problems = []

    cited = {i for g in ID_GROUP_RE.findall(text) for i in ID_RE.findall(g)}
    if bad := sorted(cited - allowed_ids):
        problems.append(f"cites IDs not in SOURCES: {bad}")
    if not cited and ans.get("where_you_stand"):
        problems.append("no source IDs cited")
    if bad := sorted({s.upper() for s in SEC_MENTION_RE.findall(text)} - allowed_secs):
        problems.append(f"mentions sections not in SOURCES: {bad}")
    if bad := sorted({w.upper() for w in SCHED_MENTION_RE.findall(text)
                      if f"Schedule {ORD_NUM[w.upper()]}" not in allowed_secs}):
        problems.append(f"mentions schedules not in SOURCES: {bad}")
    if CASE_LAW_RE.search(text):
        problems.append("appears to cite case law")
    return {"ok": not problems, "problems": problems, "cited": sorted(cited)}

def safe_fallback(sources, lang):
    lines = [f"- {c['act']}, {c['section'] if c['section'].startswith('Schedule') else 'Section ' + c['section']}: {c['title']}"
             for c in sources[:4]]
    return {"where_you_stand": "I couldn't produce an answer I can fully back up with the law I have. "
                               "These provisions look relevant — please read them or show them to a lawyer:\n" + "\n".join(lines),
            "options": [], "next_steps": ["Contact your District Legal Services Authority (DLSA) for free legal advice."],
            "what_i_dont_know": "A verified explanation for your specific situation.",
            "needs_lawyer": True, "lawyer_reason": "Automatic answer failed verification."}

In [43]:
OUT_OF_SCOPE_MSG = ("I can only help with residential tenancy problems in Bengaluru/Karnataka "
                    "(deposits, eviction, repairs, rent, notices, agreements). For other legal issues, "
                    "your District Legal Services Authority (DLSA) offers free legal advice.")

def ask(user_text, k=6):
    case = intake(user_text)
    result = {"input": user_text, "intake": case, "sources": [], "answer": None, "verification": None}

    if not case.get("in_scope", True) or case.get("topic") == "out_of_scope":
        result["answer"] = {"where_you_stand": OUT_OF_SCOPE_MSG, "options": [], "next_steps": [],
                            "what_i_dont_know": "", "needs_lawyer": False, "lawyer_reason": ""}
        return result

    missing = [r["section"] for r in case["section_refs"] if not r["in_corpus"]]
    query = case.get("english_summary") or user_text
    sources = retriever.search(query, k=k)
    result["sources"] = sources

    ans = advise(user_text, case, sources)
    check = verify(ans, sources)
    if not check["ok"]:                                       # one retry with the errors
        ans = advise(user_text, case, sources, feedback="; ".join(check["problems"]))
        check = verify(ans, sources)
    if not check["ok"]:
        ans = safe_fallback(sources, case.get("language"))

    if missing:
        ans["what_i_dont_know"] = (f"Section(s) {', '.join(missing)} that you mentioned are not in the laws I have, "
                                   "so I can't tell you what they say. " + ans.get("what_i_dont_know", ""))
    if case["escalate"]["needs_lawyer"]:                      # code overrides the model here
        ans["needs_lawyer"] = True
        ans["lawyer_reason"] = "; ".join(case["escalate"]["reasons"]) + ". " + ans.get("lawyer_reason", "")
    result.update(answer=ans, verification=check)
    return result

def show_result(r):
    a, c = r["answer"], r["intake"]
    print("=" * 100, f"\nINPUT: {r['input']}\nTOPIC: {c.get('topic')} | LANG: {c.get('language')} | "
          f"FLAGS: {c.get('urgent_flags')} | ESCALATE: {c['escalate']}")
    print("SOURCES:", [s["chunk_id"] for s in r["sources"]])
    print("VERIFIED:", r["verification"])
    print("\nWHERE YOU STAND:\n", a["where_you_stand"])
    for o in a.get("options", []):    print(" • option:", o)
    for s in a.get("next_steps", []): print(" → step:", s)
    print("DON'T KNOW:", a.get("what_i_dont_know"))
    print("NEEDS LAWYER:", a.get("needs_lawyer"), "|", a.get("lawyer_reason"))

CASES = [
    "My landlord in Koramangala cut off the water supply 3 days ago because I complained about the rent hike. What can I do?",
    "Bhai mera landlord 40,000 ka deposit wapas nahi de raha. 10 March ko flat khali kiya, agreement 11 mahine ka tha.",
    "What does Section 999 of the Karnataka Rent Act say about eviction?",
    "My husband and I want a divorce, how do I file?",
    "Landlord sent some men who threatened me and said they will throw my things out if I don't leave by Sunday.",
]
RESULTS = []
for text in CASES:
    r = ask(text)
    RESULTS.append(r)
    show_result(r)

INPUT: My landlord in Koramangala cut off the water supply 3 days ago because I complained about the rent hike. What can I do?
TOPIC: essential_services_cut | LANG: en | FLAGS: [] | ESCALATE: {'needs_lawyer': False, 'reasons': []}
SOURCES: ['KRA-s49-p1', 'KRA-s49-p2', 'KRA-s54-p4', 'KRA-s54-p3', 'KRA-s48-p1', 'KRA-s17-p2']
VERIFIED: {'ok': True, 'problems': [], 'cited': ['KRA-s49-p1', 'KRA-s49-p2', 'KRA-s54-p3']}

WHERE YOU STAND:
 You are protected by law. A landlord may not cut off water without just and sufficient cause.[KRA-s49-p1]
 • option: Apply to the Controller for an order to restore the water supply immediately.[KRA-s49-p2]
 • option: Ask the Controller to award you compensation for the loss caused by the cut off.[KRA-s49-p1]
 • option: Report the offence to the police as it is punishable under the Act.[KRA-s54-p3]
 → step: Collect any written complaints, messages or notices showing you complained about the rent hike and the water was cut off.
 → step: Prepare a written appl

In [44]:
with open(OUT_DIR / "advisor_runs.jsonl", "w", encoding="utf-8") as f:
    for r in RESULTS:
        slim = {**r, "sources": [s["chunk_id"] for s in r["sources"]]}
        f.write(json.dumps(slim, ensure_ascii=False) + "\n")

usage = pd.DataFrame(USAGE_LOG)
per_case = usage[usage.tag != "ping"]["usd"].sum() / len(RESULTS)
print(usage.groupby(["tag", "model"])[["in_tok", "out_tok", "secs", "usd"]].mean().round(4))
print(f"\nAvg cost per conversation at paid rates: ${per_case:.5f}  →  "
      f"100,000 conversations ≈ ${per_case * 100_000:,.0f}")

                            in_tok   out_tok    secs     usd
tag    model                                                
advise openai/gpt-oss-120b  2209.6  766.4000  6.6860  0.0008
intake openai/gpt-oss-20b    485.0  582.1667  1.1383  0.0002
ping   openai/gpt-oss-20b     78.0   35.0000  0.5000  0.0000

Avg cost per conversation at paid rates: $0.00104  →  100,000 conversations ≈ $104


In [45]:
#Updated Cell-16

import os, time, getpass
from datetime import datetime
from groq import Groq, BadRequestError
from dotenv import load_dotenv

load_dotenv(PROJECT / ".env")                          # GROQ_API_KEY saved earlier
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
client = Groq()

available = sorted(m.id for m in client.models.list().data)
print("Live Groq models:", available)

def pick(preferences):
    for p in preferences:
        if p in available:
            return p
    raise ValueError(f"None of {preferences} available — choose from the list above")

MODEL_BIG   = pick(["openai/gpt-oss-120b", "llama-3.3-70b-versatile"])            # answers + fact-check
MODEL_SMALL = pick(["openai/gpt-oss-20b", "llama-3.1-8b-instant", MODEL_BIG])     # intake
print("MODEL_BIG =", MODEL_BIG, "| MODEL_SMALL =", MODEL_SMALL)

# USD per 1M tokens (input, output), Groq list prices Aug 2026 — logged to estimate cost at scale
PRICES = {"openai/gpt-oss-120b": (0.15, 0.60), "openai/gpt-oss-20b": (0.075, 0.30)}
USAGE_LOG = []

def parse_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        raise ValueError(f"No JSON in model output: {text[:200]}")
    return json.loads(m.group(0))

def llm(messages, model, json_mode=False, temperature=0.2, max_tokens=3000, retries=3, tag=""):
    kwargs = dict(model=model, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    for attempt in range(retries):
        try:
            t0 = time.time()
            r = client.chat.completions.create(**kwargs)
            break
        except BadRequestError:
            # Groq rejects output that isn't valid JSON → retry as plain text and parse it ourselves
            if "response_format" in kwargs and attempt < retries - 1:
                kwargs.pop("response_format")
                print("  JSON mode rejected, retrying as plain text")
                continue
            raise
        except Exception as e:                              # rate limits etc. → back off
            if attempt == retries - 1:
                raise
            wait = 4 * (2 ** attempt)
            print(f"  LLM error ({type(e).__name__}), retrying in {wait}s")
            time.sleep(wait)
    u = r.usage
    pin, pout = PRICES.get(model, (0, 0))
    USAGE_LOG.append({"tag": tag, "model": model, "in_tok": u.prompt_tokens, "out_tok": u.completion_tokens,
                      "secs": round(time.time() - t0, 2),
                      "usd": (u.prompt_tokens * pin + u.completion_tokens * pout) / 1e6})
    text = r.choices[0].message.content or ""
    return parse_json(text) if json_mode else text

print(llm([{"role": "user", "content": "Reply with just the word OK."}], MODEL_SMALL, tag="ping"))

Live Groq models: ['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']
MODEL_BIG = openai/gpt-oss-120b | MODEL_SMALL = openai/gpt-oss-20b
OK


In [46]:
#Updated Cell 17 
TOPICS = ["deposit_refund", "eviction_threat", "essential_services_cut", "repairs",
          "rent_increase", "rent_payment_or_receipts", "agreement_registration_stamp",
          "notice_to_vacate", "holding_over", "legal_aid", "other_tenancy", "out_of_scope"]
FLAGS = ["violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice",
         "deadline_within_7_days", "commercial_premises", "outside_karnataka"]

INTAKE_SYSTEM = f"""You are the intake step of a help tool for RESIDENTIAL tenancy problems in Bengaluru / Karnataka, India.
Read the user's message and extract facts. Do NOT give legal advice. Never invent dates, years, amounts or facts:
use null or empty lists when something is not stated. Write dates exactly as precise as the user gave them.
Return ONLY a JSON object with these keys:
- "language": "en", "hi" (Hindi in Devanagari), "hinglish" (Hindi in Roman letters), "kn", or "other"
- "topic": one of {TOPICS}
- "in_scope": true only if this is a residential landlord-tenant matter (Karnataka or location unstated)
- "role": "tenant", "landlord" or "unknown"
- "english_summary": 1-3 neutral English sentences describing the problem and what the user wants
- "facts": {{"timeline": [{{"date": str|null, "event": str}}], "amounts_inr": [{{"what": str, "amount": number}}],
             "evidence": [str], "agreement": "written registered" | "written unregistered" | "oral" | null}}
- "missing_info": up to 3 short questions whose answers would change the advice
- "urgent_flags": subset of {FLAGS}"""

# Code-side red flags: these never depend on the model noticing them
RED_FLAG_RULES = {
    "violence_or_threat": r"\b(threat\w*|beat\w*|hit me|assault\w*|violen\w*|goons?|dhamki|maar\w*)\b",
    "locked_out_or_belongings_removed": r"\b(locked (me )?out|changed the locks?|threw (out )?my|removed my (things|belongings))\b",
    "court_or_police_notice": r"\b(summons|court notice|fir|police|legal notice|lawyer'?s? notice|vakil)\b",
}
LAWYER_FLAGS = {"violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice"}
LARGE_AMOUNT_INR = 500_000

def fix_invented_years(case, user_text):
    """Code check: if the model added a year the user never wrote, remove it."""
    for ev in (case.get("facts") or {}).get("timeline") or []:
        d = ev.get("date") or ""
        m = re.match(r"((?:19|20)\d{2})-(\d{2})-(\d{2})$", d)
        if m and m.group(1) not in user_text:
            ev["date"] = datetime.strptime(d, "%Y-%m-%d").strftime("%d %B") + " (year not stated)"

def intake(user_text):
    data = llm([{"role": "system", "content": INTAKE_SYSTEM},
                {"role": "user", "content": user_text}], MODEL_SMALL, json_mode=True, tag="intake")
    fix_invented_years(data, user_text)

    flags = set(data.get("urgent_flags") or [])
    for flag, pattern in RED_FLAG_RULES.items():
        if re.search(pattern, user_text, re.I):
            flags.add(flag)
    data["urgent_flags"] = sorted(flags)
    data["section_refs"] = check_section_refs(user_text)          # Cell 14

    amounts = [a.get("amount") or 0 for a in (data.get("facts") or {}).get("amounts_inr") or []]
    reasons = [f for f in data["urgent_flags"] if f in LAWYER_FLAGS]
    if amounts and max(amounts) >= LARGE_AMOUNT_INR:
        reasons.append(f"amount at stake ≥ ₹{LARGE_AMOUNT_INR:,}")
    data["escalate"] = {"needs_lawyer": bool(reasons), "reasons": reasons}
    return data

demo = intake("Bhai mera landlord 40,000 ka deposit wapas nahi de raha. Maine 10 March ko flat khali kiya tha, "
              "agreement 11 mahine ka tha. Ab phone bhi nahi utha raha.")
print(json.dumps(demo, indent=2, ensure_ascii=False))

{
  "language": "hinglish",
  "topic": "deposit_refund",
  "in_scope": true,
  "role": "tenant",
  "english_summary": "The tenant claims the landlord has not returned a ₹40,000 deposit after vacating the flat on 10 March. The tenancy agreement was for 11 months and the landlord is unresponsive to phone calls.",
  "facts": {
    "timeline": [
      {
        "date": "10 March",
        "event": "Tenant vacated the flat"
      }
    ],
    "amounts_inr": [
      {
        "what": "deposit",
        "amount": 40000
      }
    ],
    "evidence": [],
    "agreement": null
  },
  "missing_info": [
    "When did the tenancy start?",
    "Did the tenant give written notice before vacating?",
    "Has the tenant sent any written communication to the landlord regarding the deposit?"
  ],
  "urgent_flags": [],
  "section_refs": [],
  "escalate": {
    "needs_lawyer": false,
    "reasons": []
  }
}


In [66]:
#Newly Updated Cell 17 

TOPICS = ["deposit_refund", "eviction_threat", "essential_services_cut", "repairs",
          "rent_increase", "rent_payment_or_receipts", "agreement_registration_stamp",
          "notice_to_vacate", "holding_over", "legal_aid", "other_tenancy", "out_of_scope"]
FLAGS = ["violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice",
         "deadline_within_7_days", "commercial_premises", "outside_karnataka"]

INTAKE_SYSTEM = f"""You are the intake step of a help tool for RESIDENTIAL tenancy problems in Bengaluru / Karnataka, India.
Read the user's message and extract facts. Do NOT give legal advice. Never invent dates, years, amounts or facts:
use null or empty lists when something is not stated. Write dates exactly as precise as the user gave them.
Return ONLY a JSON object with these keys:
- "language": "en", "hi" (Hindi in Devanagari), "hinglish" (Hindi in Roman letters), "kn", or "other"
- "topic": one of {TOPICS}
- "in_scope": true only if this is a residential landlord-tenant matter (Karnataka or location unstated)
- "role": "tenant", "landlord" or "unknown"
- "english_summary": 1-3 neutral English sentences describing the problem and what the user wants
- "facts": {{"timeline": [{{"date": str|null, "event": str}}], "amounts_inr": [{{"what": str, "amount": number}}],
             "evidence": [str], "agreement": "written registered" | "written unregistered" | "oral" | null}}
- "missing_info": up to 3 short questions whose answers would change the advice
- "urgent_flags": subset of {FLAGS}"""

# Code-side red flags: these never depend on the model noticing them
RED_FLAG_RULES = {
    "violence_or_threat": r"\b(threat\w*|beat\w*|hit me|assault\w*|violen\w*|goons?|dhamki|maar\w*)\b",
    "locked_out_or_belongings_removed": r"\b(locked (me )?out|changed the locks?|threw (out )?my|removed my (things|belongings))\b",
    "court_or_police_notice": r"\b(summons|court notice|fir|police|legal notice|lawyer'?s? notice|vakil)\b",
}
LAWYER_FLAGS = {"violence_or_threat", "locked_out_or_belongings_removed", "court_or_police_notice"}
LARGE_AMOUNT_INR = 500_000

def fix_invented_years(case, user_text):
    """Code check: if the model added a year the user never wrote, remove it."""
    for ev in (case.get("facts") or {}).get("timeline") or []:
        d = ev.get("date") or ""
        m = re.match(r"((?:19|20)\d{2})-(\d{2})-(\d{2})$", d)
        if m and m.group(1) not in user_text:
            ev["date"] = datetime.strptime(d, "%Y-%m-%d").strftime("%d %B") + " (year not stated)"

def guess_language(text):
    """Script-based guess, used only when the intake model fails."""
    if re.search(r"[\u0C80-\u0CFF]", text): return "kn"
    if re.search(r"[\u0900-\u097F]", text): return "hi"
    return "en"

def default_case(user_text):
    """Safe defaults if the intake model crashes or returns bad JSON."""
    return {"language": guess_language(user_text), "topic": "other_tenancy", "in_scope": True, "role": "unknown",
            "english_summary": user_text, "facts": {"timeline": [], "amounts_inr": [], "evidence": [], "agreement": None},
            "missing_info": [], "urgent_flags": [], "intake_failed": True}

def intake(user_text):
    try:
        data = llm([{"role": "system", "content": INTAKE_SYSTEM},
                    {"role": "user", "content": user_text}], MODEL_SMALL, json_mode=True, tag="intake")
        if not isinstance(data, dict) or "topic" not in data:
            raise ValueError("intake JSON missing keys")
    except Exception as e:
        print(f"  intake failed ({type(e).__name__}) → safe defaults")
        data = default_case(user_text)
    fix_invented_years(data, user_text)

    flags = set(data.get("urgent_flags") or [])
    for flag, pattern in RED_FLAG_RULES.items():
        if re.search(pattern, user_text, re.I):
            flags.add(flag)
    data["urgent_flags"] = sorted(flags)
    data["section_refs"] = check_section_refs(user_text)          # Cell 14

    amounts = [a.get("amount") or 0 for a in (data.get("facts") or {}).get("amounts_inr") or []]
    reasons = [f for f in data["urgent_flags"] if f in LAWYER_FLAGS]
    if amounts and max(amounts) >= LARGE_AMOUNT_INR:
        reasons.append(f"amount at stake ≥ ₹{LARGE_AMOUNT_INR:,}")
    data["escalate"] = {"needs_lawyer": bool(reasons), "reasons": reasons}
    return data

print("intake() ready")

intake() ready


In [47]:
#Updated Cell-18
LANG_NAMES = {"en": "English", "hi": "Hindi (Devanagari)",
              "hinglish": "Hinglish (Hindi in Roman letters, exactly as the user writes)",
              "kn": "Kannada", "other": "English"}

ADVISOR_SYSTEM = """You help ordinary people in Bengaluru understand residential tenancy problems.
STRICT RULES:
1. Use ONLY the SOURCES given. Every legal statement must end with its source ID in square brackets, e.g. [KRA-s49-p1].
2. Never mention any Act, section, schedule, rule, court case or judgment that is not in the SOURCES. Do not cite case law at all.
3. If the SOURCES do not answer the question, say so plainly in "what_i_dont_know". Never fill gaps from memory.
4. Sources marked "summary" are plain-language summaries, not the bare Act: say "(summary)" when you rely on them.
5. Remind the user that their rent agreement's terms matter and the Act may not apply to every rental.
6. Plain words, short sentences, no legal jargon. Write the answer in: {language}.
7. Many sections apply only in specific situations (e.g. only after a court eviction order, only to rent paid in advance).
   Before relying on a section, check its conditions against the user's facts. If the conditions don't match,
   do not use that section; say instead that the laws provided don't directly cover the situation.
Return ONLY a JSON object:
{{"where_you_stand": str, "options": [str], "next_steps": [str], "what_i_dont_know": str,
  "needs_lawyer": bool, "lawyer_reason": str}}"""

def format_sources(chunks):
    blocks = []
    for c in chunks:
        label = c["section"] if c["section"].startswith("Schedule") else f"Section {c['section']}"
        kind = " (summary, not bare text)" if c["status"] == "commentary" else ""
        blocks.append(f"[{c['chunk_id']}] {c['act']}, {label}{kind}: {c['title']}\n{c['text']}")
    return "\n\n".join(blocks)

def advise(user_text, case, sources, feedback=None):
    msgs = [{"role": "system", "content": ADVISOR_SYSTEM.format(language=LANG_NAMES.get(case.get("language"), "English"))},
            {"role": "user", "content":
                f"USER MESSAGE:\n{user_text}\n\nEXTRACTED FACTS:\n{json.dumps(case.get('facts'), ensure_ascii=False)}\n\n"
                f"SOURCES:\n{format_sources(sources)}"}]
    if feedback:
        msgs.append({"role": "user", "content": f"Your previous answer failed checks: {feedback}\n"
                                                 "Rewrite it following the rules exactly. Remove or correct the failing claims."})
    return llm(msgs, MODEL_BIG, json_mode=True, temperature=0.1, tag="advise")

In [48]:
#Cell 19 Updated 

# ---------- 1. Citation verifier (code): are cited IDs / sections really in the sources? ----------
ID_GROUP_RE = re.compile(r"\[([A-Z]{3}-[^\]]+)\]")
ID_RE = re.compile(r"[A-Z]{3}-(?:s\d{1,3}[A-Z]{0,3}|sch\d+)(?:-p\d+)?")
SEC_MENTION_RE = re.compile(r"\b(?:section|sec\.?|s\.)\s*(\d{1,3}[A-Z]?)\b", re.I)
SCHED_MENTION_RE = re.compile(rf"\b({ORD})\s+schedule\b", re.I)
CASE_LAW_RE = re.compile(r"\b(v\.|vs\.?|versus)\s+[A-Z]|\bAIR\s+\d{4}|\(\d{4}\)\s*\d+\s*SCC")

def answer_text(ans):
    return " ".join([ans.get("where_you_stand", ""), *ans.get("options", []),
                     *ans.get("next_steps", []), ans.get("what_i_dont_know", "")])

def verify(ans, sources):
    text = answer_text(ans)
    allowed_ids = {c["chunk_id"] for c in sources}
    allowed_secs = {c["section"] for c in sources}
    problems = []

    cited = {i for g in ID_GROUP_RE.findall(text) for i in ID_RE.findall(g)}
    if bad := sorted(cited - allowed_ids):
        problems.append(f"cites IDs not in SOURCES: {bad}")
    if not cited and ans.get("where_you_stand"):
        problems.append("no source IDs cited")
    if bad := sorted({s.upper() for s in SEC_MENTION_RE.findall(text)} - allowed_secs):
        problems.append(f"mentions sections not in SOURCES: {bad}")
    if bad := sorted({w.upper() for w in SCHED_MENTION_RE.findall(text)
                      if f"Schedule {ORD_NUM[w.upper()]}" not in allowed_secs}):
        problems.append(f"mentions schedules not in SOURCES: {bad}")
    if CASE_LAW_RE.search(text):
        problems.append("appears to cite case law")
    return {"ok": not problems, "problems": problems, "cited": sorted(cited)}

# ---------- 2. Support checker (AI): does the cited law actually apply to THESE facts? ----------
SUPPORT_SYSTEM = """You are a strict legal fact-checker. You get the user's FACTS, the SOURCES (law text),
and numbered CLAIMS from an answer, each with the source IDs it cites. For each claim decide:
- "supported": the cited source says this AND its conditions fit the user's facts
- "conditions_not_met": the source says something similar, but only for a situation that does not match
   the user's facts (e.g. applies only when the landlord recovers possession through court)
- "unsupported": the cited source does not say this
Return ONLY JSON: {"claims": [{"i": int, "verdict": str, "reason": str}]}"""

def split_claims(ans):
    pieces = (re.split(r"(?<=[.!?।\]])\s+", ans.get("where_you_stand", ""))
              + ans.get("options", []) + ans.get("next_steps", []))
    claims = []
    for p in pieces:
        ids = [i for g in ID_GROUP_RE.findall(p) for i in ID_RE.findall(g)]
        if ids:
            claims.append({"text": p.strip(), "ids": ids})
    return claims

def check_support(ans, sources, case):
    claims = split_claims(ans)
    if not claims:
        return {"ok": True, "problems": [], "verdicts": []}
    cited = {i for c in claims for i in c["ids"]}
    src = [s for s in sources if s["chunk_id"] in cited]
    payload = (f"FACTS:\n{json.dumps(case.get('facts'), ensure_ascii=False)}\n"
               f"USER SUMMARY: {case.get('english_summary')}\n\nSOURCES:\n{format_sources(src)}\n\nCLAIMS:\n" +
               "\n".join(f"{n}. {c['text']}  (cites {c['ids']})" for n, c in enumerate(claims)))
    out = llm([{"role": "system", "content": SUPPORT_SYSTEM}, {"role": "user", "content": payload}],
              MODEL_BIG, json_mode=True, temperature=0, tag="support")
    verdicts = out.get("claims", [])
    problems = [f"claim {v['i']} ('{claims[v['i']]['text'][:70]}…') is {v['verdict']}: {v['reason']}"
                for v in verdicts
                if v.get("verdict") != "supported" and isinstance(v.get("i"), int) and 0 <= v["i"] < len(claims)]
    return {"ok": not problems, "problems": problems, "verdicts": verdicts}

# ---------- 3. Safe fallback when checks keep failing ----------
def safe_fallback(sources, lang):
    lines = [f"- {c['act']}, {c['section'] if c['section'].startswith('Schedule') else 'Section ' + c['section']}: {c['title']}"
             for c in sources[:4]]
    return {"where_you_stand": "I couldn't produce an answer I can fully back up with the law I have. "
                               "These provisions may be relevant — please read them or show them to a lawyer:\n" + "\n".join(lines),
            "options": [], "next_steps": ["Contact your District Legal Services Authority (DLSA) for free legal advice."],
            "what_i_dont_know": "A verified explanation for your specific situation.",
            "needs_lawyer": True, "lawyer_reason": "Automatic answer failed verification."}

In [49]:
#Updated Cell 20 

OUT_OF_SCOPE_MSG = ("I can only help with residential tenancy problems in Bengaluru/Karnataka "
                    "(deposits, eviction, repairs, rent, notices, agreements). For other legal issues, "
                    "your District Legal Services Authority (DLSA) offers free legal advice.")

def section_not_found_answer(case, query):
    """Code-only reply when the user asks about a section that isn't in the corpus."""
    refs = case["section_refs"]
    secs = ", ".join(r["section"] for r in refs)
    cover = []
    for code in refs[0]["acts_checked"]:
        nums = sorted(int(re.match(r"\d+", s).group()) for s in KNOWN_SECTIONS[code] if re.match(r"\d+", s))
        if nums:
            cover.append(f"{code}: sections {nums[0]}–{nums[-1]}")
    top = retriever.search(query, k=3)
    return {"where_you_stand": f"I couldn't find Section {secs} in the laws I have ({'; '.join(cover)}), "
                               "so I won't guess what it says.",
            "options": [f"Did you mean {t['act']}, Section {t['section']}: {t['title']}?" for t in top
                        if not t["section"].startswith("Schedule")],
            "next_steps": ["Check the section number on your notice or document and ask again."],
            "what_i_dont_know": f"The text of Section {secs}.", "needs_lawyer": False, "lawyer_reason": ""}

def ask(user_text, k=6):
    case = intake(user_text)
    result = {"input": user_text, "intake": case, "sources": [], "answer": None,
              "verification": None, "support": None, "attempts": 0}

    # 1. Out of scope → fixed reply, no retrieval
    if not case.get("in_scope", True) or case.get("topic") == "out_of_scope":
        result["answer"] = {"where_you_stand": OUT_OF_SCOPE_MSG, "options": [], "next_steps": [],
                            "what_i_dont_know": "", "needs_lawyer": False, "lawyer_reason": ""}
        return result

    # 2. Asked about a section we don't have → code answers, no LLM
    query = case.get("english_summary") or user_text
    refs = case["section_refs"]
    if refs and all(not r["in_corpus"] for r in refs):
        result["answer"] = section_not_found_answer(case, query)
        return result

    # 3. Retrieve → advise → verify citations → fact-check support (max 2 attempts)
    sources = retriever.search(query, k=k)
    result["sources"] = sources
    feedback, check, support = None, None, None
    for attempt in range(2):
        result["attempts"] = attempt + 1
        ans = advise(user_text, case, sources, feedback)
        check = verify(ans, sources)
        support = check_support(ans, sources, case) if check["ok"] else None
        if check["ok"] and support["ok"]:
            break
        feedback = "; ".join(check["problems"] + (support["problems"] if support else []))
    else:
        ans = safe_fallback(sources, case.get("language"))

    # 4. Code overrides the model on escalation
    if case["escalate"]["needs_lawyer"]:
        ans["needs_lawyer"] = True
        ans["lawyer_reason"] = "; ".join(case["escalate"]["reasons"]) + ". " + ans.get("lawyer_reason", "")
    result.update(answer=ans, verification=check, support=support)
    return result

def show_result(r):
    a, c = r["answer"], r["intake"]
    print("=" * 100, f"\nINPUT: {r['input']}\nTOPIC: {c.get('topic')} | LANG: {c.get('language')} | "
          f"FLAGS: {c.get('urgent_flags')} | ESCALATE: {c['escalate']}")
    print("TIMELINE:", (c.get("facts") or {}).get("timeline"))
    print("SOURCES:", [s["chunk_id"] for s in r["sources"]], "| ATTEMPTS:", r["attempts"])
    print("VERIFIED:", r["verification"])
    print("SUPPORT:", r["support"] and {"ok": r["support"]["ok"], "problems": r["support"]["problems"]})
    print("\nWHERE YOU STAND:\n", a["where_you_stand"])
    for o in a.get("options", []):    print(" • option:", o)
    for s in a.get("next_steps", []): print(" → step:", s)
    print("DON'T KNOW:", a.get("what_i_dont_know"))
    print("NEEDS LAWYER:", a.get("needs_lawyer"), "|", a.get("lawyer_reason"))

CASES = [
    "My landlord in Koramangala cut off the water supply 3 days ago because I complained about the rent hike. What can I do?",
    "Bhai mera landlord 40,000 ka deposit wapas nahi de raha. 10 March ko flat khali kiya, agreement 11 mahine ka tha.",
    "What does Section 999 of the Karnataka Rent Act say about eviction?",
    "My husband and I want a divorce, how do I file?",
    "Landlord sent some men who threatened me and said they will throw my things out if I don't leave by Sunday.",
]
USAGE_LOG.clear()
RESULTS = []
for text in CASES:
    r = ask(text)
    RESULTS.append(r)
    show_result(r)

INPUT: My landlord in Koramangala cut off the water supply 3 days ago because I complained about the rent hike. What can I do?
TOPIC: essential_services_cut | LANG: en | FLAGS: [] | ESCALATE: {'needs_lawyer': False, 'reasons': []}
TIMELINE: [{'date': None, 'event': 'water supply cut off by landlord'}]
SOURCES: ['KRA-s49-p1', 'KRA-s49-p2', 'KRA-s54-p4', 'KRA-s17-p2', 'KRA-s54-p3', 'KRA-s17-p3'] | ATTEMPTS: 1
VERIFIED: {'ok': True, 'problems': [], 'cited': ['KRA-s49-p1', 'KRA-s54-p3']}
SUPPORT: {'ok': True, 'problems': []}

WHERE YOU STAND:
 Your landlord cut off water, an essential supply, without just and sufficient cause. This breaches Section 49 of the Karnataka Rent Act [KRA-s49-p1].
 • option: Apply to the Rent Controller under Section 49(2) for an order to restore water immediately (interim order) [KRA-s49-p1].
 • option: Ask the Controller to direct the landlord to pay compensation for the wrongful cut‑off [KRA-s49-p1].
 • option: Seek penalty against the landlord under Section 5

In [67]:
#Newly Updated Cell 20 
OUT_OF_SCOPE_MSG = ("I can only help with residential tenancy problems in Bengaluru/Karnataka "
                    "(deposits, eviction, repairs, rent, notices, agreements). For other legal issues, "
                    "your District Legal Services Authority (DLSA) offers free legal advice.")

def section_not_found_answer(case, query):
    """Code-only reply when the user asks about a section that isn't in the corpus."""
    refs = case["section_refs"]
    secs = ", ".join(r["section"] for r in refs)
    cover = []
    for code in refs[0]["acts_checked"]:
        nums = sorted(int(re.match(r"\d+", s).group()) for s in KNOWN_SECTIONS[code] if re.match(r"\d+", s))
        if nums:
            cover.append(f"{code}: sections {nums[0]}–{nums[-1]}")
    seen, opts = set(), []
    for t in retriever.search(query, k=6):
        key = (t["act_code"], t["section"])
        if key in seen or t["section"].startswith("Schedule"):
            continue
        seen.add(key)
        opts.append(f"Did you mean {t['act']}, Section {t['section']}: {t['title']}?")
        if len(opts) == 3:
            break
    return {"where_you_stand": f"I couldn't find Section {secs} in the laws I have ({'; '.join(cover)}), "
                               "so I won't guess what it says.",
            "options": opts,
            "next_steps": ["Check the section number on your notice or document and ask again."],
            "what_i_dont_know": f"The text of Section {secs}.", "needs_lawyer": False, "lawyer_reason": ""}

def ask(user_text, k=6):
    # 0. Fake / uncovered section → answered by code BEFORE any LLM call
    refs = check_section_refs(user_text)
    if refs and all(not r["in_corpus"] for r in refs):
        case = {**default_case(user_text), "intake_failed": False, "topic": None,
                "section_refs": refs, "escalate": {"needs_lawyer": False, "reasons": []}}
        return {"input": user_text, "intake": case, "sources": [], "verification": None, "support": None,
                "attempts": 0, "answer": section_not_found_answer(case, user_text)}

    case = intake(user_text)
    result = {"input": user_text, "intake": case, "sources": [], "answer": None,
              "verification": None, "support": None, "attempts": 0}

    # 1. Out of scope → fixed reply, no retrieval
    if not case.get("in_scope", True) or case.get("topic") == "out_of_scope":
        result["answer"] = {"where_you_stand": OUT_OF_SCOPE_MSG, "options": [], "next_steps": [],
                            "what_i_dont_know": "", "needs_lawyer": False, "lawyer_reason": ""}
        return result

    # 2. Retrieve → advise → verify citations → fact-check support (max 2 attempts)
    query = case.get("english_summary") or user_text
    sources = retriever.search(query, k=k)
    result["sources"] = sources
    feedback, check, support = None, None, None
    for attempt in range(2):
        result["attempts"] = attempt + 1
        ans = advise(user_text, case, sources, feedback)
        check = verify(ans, sources)
        support = check_support(ans, sources, case) if check["ok"] else None
        if check["ok"] and support["ok"]:
            break
        feedback = "; ".join(check["problems"] + (support["problems"] if support else []))
    else:
        ans = safe_fallback(sources, case.get("language"))

    # 3. Code overrides the model on escalation
    if case["escalate"]["needs_lawyer"]:
        ans["needs_lawyer"] = True
        ans["lawyer_reason"] = "; ".join(case["escalate"]["reasons"]) + ". " + ans.get("lawyer_reason", "")
    result.update(answer=ans, verification=check, support=support)
    return result

def show_result(r):
    a, c = r["answer"], r["intake"]
    print("=" * 100, f"\nINPUT: {r['input']}\nTOPIC: {c.get('topic')} | LANG: {c.get('language')} | "
          f"FLAGS: {c.get('urgent_flags')} | ESCALATE: {c['escalate']}")
    print("TIMELINE:", (c.get("facts") or {}).get("timeline"))
    print("SOURCES:", [s["chunk_id"] for s in r["sources"]], "| ATTEMPTS:", r["attempts"])
    print("VERIFIED:", r["verification"])
    print("SUPPORT:", r["support"] and {"ok": r["support"]["ok"], "problems": r["support"]["problems"]})
    print("\nWHERE YOU STAND:\n", a["where_you_stand"])
    for o in a.get("options", []):    print(" • option:", o)
    for s in a.get("next_steps", []): print(" → step:", s)
    print("DON'T KNOW:", a.get("what_i_dont_know"))
    print("NEEDS LAWYER:", a.get("needs_lawyer"), "|", a.get("lawyer_reason"))

# quick no-cost check: fake section is answered by code
show_result(ask("What does Section 999 of the Karnataka Rent Act say about eviction?"))

INPUT: What does Section 999 of the Karnataka Rent Act say about eviction?
TOPIC: None | LANG: en | FLAGS: [] | ESCALATE: {'needs_lawyer': False, 'reasons': []}
TIMELINE: []
SOURCES: [] | ATTEMPTS: 0
VERIFIED: None
SUPPORT: None

WHERE YOU STAND:
 I couldn't find Section 999 in the laws I have (KRA: sections 1–70), so I won't guess what it says.
 • option: Did you mean Karnataka Rent Act, 1999, Section 27: Protection of tenants against eviction?
 • option: Did you mean Karnataka Rent Act, 1999, Section 34: Sub-tenant to be tenant in certain cases?
 • option: Did you mean Karnataka Rent Act, 1999, Section 45: Deposit and payment of rent during the pendency of proceedings for eviction?
 → step: Check the section number on your notice or document and ask again.
DON'T KNOW: The text of Section 999.
NEEDS LAWYER: False | 


In [50]:
with open(OUT_DIR / "advisor_runs.jsonl", "w", encoding="utf-8") as f:
    for r in RESULTS:
        slim = {**r, "sources": [s["chunk_id"] for s in r["sources"]]}
        f.write(json.dumps(slim, ensure_ascii=False) + "\n")
print("saved:", OUT_DIR / "advisor_runs.jsonl")

usage = pd.DataFrame(USAGE_LOG)
print("\nPer-call averages by step:")
print(usage.groupby(["tag", "model"])[["in_tok", "out_tok", "secs", "usd"]].mean().round(4))

total_usd = usage["usd"].sum()
total_secs = usage["secs"].sum()
n = len(RESULTS)
print(f"\nCalls: {len(usage)} for {n} conversations")
print(f"Avg cost per conversation (paid rates): ${total_usd / n:.5f}  →  100,000 conversations ≈ ${total_usd / n * 100_000:,.0f}")
print(f"Avg LLM time per conversation: {total_secs / n:.1f}s")

saved: /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/advisor_runs.jsonl

Per-call averages by step:
                             in_tok  out_tok    secs     usd
tag     model                                               
advise  openai/gpt-oss-120b  2423.6    883.2  16.034  0.0009
intake  openai/gpt-oss-20b    497.4    656.0   1.140  0.0002
support openai/gpt-oss-120b  1082.4    495.8   8.956  0.0005

Calls: 15 for 5 conversations
Avg cost per conversation (paid rates): $0.00159  →  100,000 conversations ≈ $159
Avg LLM time per conversation: 26.1s


In [51]:
# gold: list of (act_code, section) — section None = any section of that act is acceptable
TESTSET = [
    # ---- grounded: law clearly covers it ----
    dict(id="G01", cat="grounded", lang="en", text="My landlord in Koramangala cut off the water supply 3 days ago because I complained about the rent hike. What can I do?",
         topics={"essential_services_cut"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","49")],
         expect="Explains that cutting off water without just cause is not allowed and the tenant can apply to the Controller for restoration."),
    dict(id="G02", cat="grounded", lang="en", text="My landlord takes rent in cash every month but never gives me a receipt.",
         topics={"rent_payment_or_receipts"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","16")],
         expect="Says the tenant is entitled to a written receipt for rent paid."),
    dict(id="G03", cat="grounded", lang="en", text="My landlord is refusing to accept this month's rent so he can later claim I defaulted. What should I do?",
         topics={"rent_payment_or_receipts"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","17")],
         expect="Explains the tenant can deposit the rent (with the Controller) when the landlord refuses to accept it."),
    dict(id="G04", cat="grounded", lang="en", text="My landlord says he needs the flat for his son and I must leave within a month. Can he just do that?",
         topics={"eviction_threat", "notice_to_vacate"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","27")],
         expect="Explains eviction needs a Court order on grounds listed in the Act (own/family use is one ground); the landlord cannot simply force the tenant out."),
    dict(id="G05", cat="grounded", lang="en", text="The roof has been leaking for two months. I told the landlord twice but he won't fix it.",
         topics={"repairs"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","47"), ("KRA","Schedule 5"), ("TPA","108")],
         expect="Says structural repairs are the landlord's duty and explains the tenant's remedies."),
    dict(id="G06", cat="grounded", lang="en", text="Who should pay to replace a broken tap and clean the drain, me or the owner?",
         topics={"repairs"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","Schedule 5"), ("KRA","48")],
         expect="Says tap and drain cleaning are day-to-day repairs that fall on the tenant."),
    dict(id="G07", cat="grounded", lang="en", text="There is no written agreement and my landlord gave me only 7 days' notice to vacate. Is that enough?",
         topics={"notice_to_vacate", "eviction_threat"}, in_scope=True, mode="answer", escalate=False, gold=[("TPA","106"), ("KRA","27")],
         expect="Mentions that a month-to-month tenancy ordinarily needs 15 days' written notice and/or that eviction needs legal grounds."),
    dict(id="G08", cat="grounded", lang="en", text="We signed a 2-year rental agreement but never registered it. Is it valid?",
         topics={"agreement_registration_stamp"}, in_scope=True, mode="answer", escalate=False, gold=[("TPA","107"), ("REG","17")],
         expect="Says leases over one year must be made by a registered instrument."),
    dict(id="G09", cat="grounded", lang="en", text="My 11-month lease ended in June but I stayed on and the landlord kept taking rent. Am I still a tenant?",
         topics={"holding_over"}, in_scope=True, mode="answer", escalate=False, gold=[("TPA","116")],
         expect="Explains holding over: accepting rent after expiry renews the tenancy month to month."),
    dict(id="G10", cat="grounded", lang="en", text="How much stamp duty do I need to pay on a rental agreement in Karnataka?",
         topics={"agreement_registration_stamp"}, in_scope=True, mode="answer", escalate=False, gold=[("KSA", None)],
         expect="Relies on the Karnataka Stamp Act; does not invent a figure not present in the sources."),
    dict(id="G11", cat="grounded", lang="en", text="I can't afford a lawyer. Can I get free legal help for my landlord problem?",
         topics={"legal_aid"}, in_scope=True, mode="answer", escalate=False, gold=[("LSA", None)],
         expect="Explains free legal services exist under the Legal Services Authorities Act and who is eligible."),
    # ---- coverage gaps / myths ----
    dict(id="M01", cat="gap_or_myth", lang="hinglish", text="Bhai mera landlord 40,000 ka deposit wapas nahi de raha. 10 March ko flat khali kiya, agreement 11 mahine ka tha.",
         topics={"deposit_refund"}, in_scope=True, mode="answer", escalate=False, gold=[], must_not=["90 din", "90 days", "12%"],
         expect="Does NOT claim a statutory 90-day refund or 12% interest; is honest that the law provided doesn't directly cover a voluntary-vacating deposit refund; gives practical steps."),
    dict(id="M02", cat="gap_or_myth", lang="en", text="I heard the Karnataka Rent Act caps the security deposit at 2 months' rent, so my landlord must refund the rest. Right?",
         topics={"deposit_refund"}, in_scope=True, mode="answer", escalate=False, gold=[],
         expect="Does NOT confirm a 2-month deposit cap; says it cannot find such a rule in the law provided."),
    dict(id="M03", cat="gap_or_myth", lang="en", text="My landlord deducted 25,000 from my 60,000 deposit for repainting the whole flat after I lived there 2 years.",
         topics={"deposit_refund", "repairs"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","Schedule 5"), ("KRA","47")], must_not=["12%"],
         expect="Either notes that periodic whitewashing/painting is listed as a landlord duty, or honestly says the law provided doesn't settle deductions; no invented rules."),
    # ---- escalation ----
    dict(id="E01", cat="escalation", lang="en", text="Landlord sent some men who threatened me and said they will throw my things out if I don't leave by Sunday.",
         topics={"eviction_threat"}, in_scope=True, mode="answer", escalate=True, gold=[("KRA","27")],
         expect="Urges contacting a lawyer/DLSA (and police for safety); does not invent criminal law sections."),
    dict(id="E02", cat="escalation", lang="en", text="I came back from my village and the landlord had changed the locks and kept my things inside.",
         topics={"eviction_threat"}, in_scope=True, mode="answer", escalate=True, gold=[("KRA","27")],
         expect="Flags urgency and recommends a lawyer/DLSA."),
    dict(id="E03", cat="escalation", lang="en", text="I received a summons from the court for an eviction case filed by my landlord. The hearing is next week.",
         topics={"eviction_threat"}, in_scope=True, mode="answer", escalate=True, gold=[("KRA","27")],
         expect="Strongly recommends a lawyer/DLSA before the hearing; explains eviction grounds at a high level only."),
    dict(id="E04", cat="escalation", lang="en", text="I gave my landlord Rs 8,00,000 as lease (bhogya) amount for our house and now he refuses to return it after the lease ended.",
         topics={"deposit_refund", "other_tenancy"}, in_scope=True, mode="answer", escalate=True, gold=[],
         expect="Recommends a lawyer given the large amount; no invented refund rules."),
    # ---- fake sections ----
    dict(id="F01", cat="fake_section", lang="en", text="What does Section 999 of the Karnataka Rent Act say about eviction?",
         topics=None, in_scope=True, mode="section_not_found", escalate=False, gold=[],
         expect="Says Section 999 is not in its laws and does not invent its content."),
    dict(id="F02", cat="fake_section", lang="en", text="What does Section 250 of the Transfer of Property Act say about tenants?",
         topics=None, in_scope=True, mode="section_not_found", escalate=False, gold=[],
         expect="Says it doesn't have Section 250 and does not invent its content."),
    # ---- out of scope ----
    dict(id="O01", cat="out_of_scope", lang="en", text="My husband and I want a divorce, how do I file?",
         topics={"out_of_scope"}, in_scope=False, mode="out_of_scope", escalate=False, gold=[],
         expect="Politely declines and points to DLSA."),
    dict(id="O02", cat="out_of_scope", lang="en", text="My employer has not paid my salary for 3 months. What can I do?",
         topics={"out_of_scope"}, in_scope=False, mode="out_of_scope", escalate=False, gold=[],
         expect="Politely declines (not tenancy) and points to DLSA."),
    dict(id="O03", cat="out_of_scope", lang="en", text="Someone is selling a plot of land that I own using fake papers.",
         topics={"out_of_scope"}, in_scope=False, mode="out_of_scope", escalate=False, gold=[],
         expect="Declines: property ownership disputes are out of scope."),
    # ---- multilingual ----
    dict(id="L01", cat="multilingual", lang="hi", text="मकान मालिक ने बिजली काट दी है क्योंकि मैंने किराया बढ़ाने से मना किया।",
         topics={"essential_services_cut"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","49")],
         expect="Answers in Hindi; explains cutting off electricity is not allowed and the Controller route."),
    dict(id="L02", cat="multilingual", lang="kn", text="ಮನೆ ಮಾಲೀಕರು ನನ್ನನ್ನು ಮನೆಯಿಂದ ಹೊರಹಾಕಲು ಬಯಸುತ್ತಾರೆ, ಆದರೆ ನಾನು ಪ್ರತಿ ತಿಂಗಳು ಬಾಡಿಗೆ ಕೊಡುತ್ತಿದ್ದೇನೆ.",
         topics={"eviction_threat", "notice_to_vacate"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","27")],
         expect="Answers in Kannada; explains eviction needs legal grounds/Court order."),
    dict(id="L03", cat="multilingual", lang="kn", text="ಮನೆಯ ಛಾವಣಿ ಸೋರುತ್ತಿದೆ, ಮಾಲೀಕರು ಸರಿಪಡಿಸುತ್ತಿಲ್ಲ.",
         topics={"repairs"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","47"), ("KRA","Schedule 5"), ("TPA","108")],
         expect="Answers in Kannada; says structural repairs are the landlord's duty."),
    dict(id="L04", cat="multilingual", lang="hinglish", text="Landlord rent ki receipt nahi deta, cash mein leta hai. Kya kar sakte hain?",
         topics={"rent_payment_or_receipts"}, in_scope=True, mode="answer", escalate=False, gold=[("KRA","16")],
         expect="Answers in Hinglish; explains right to a written receipt."),
]
print(len(TESTSET), "cases |", collections.Counter(c["cat"] for c in TESTSET))

27 cases | Counter({'grounded': 11, 'escalation': 4, 'multilingual': 4, 'gap_or_myth': 3, 'out_of_scope': 3, 'fake_section': 2})


In [68]:
RUNS_FILE = OUT_DIR / "eval_runs.jsonl"
SLEEP_BETWEEN = 3          # seconds; raise if you hit Groq rate limits

def result_mode(r):
    a = r["answer"]["where_you_stand"]
    if not r["sources"] and a == OUT_OF_SCOPE_MSG:
        return "out_of_scope"
    if a.startswith("I couldn't find Section"):
        return "section_not_found"
    return "answer"

def is_fallback(r):
    return r["answer"]["where_you_stand"].startswith("I couldn't produce an answer")

def cited_ids(r):
    return sorted({i for g in ID_GROUP_RE.findall(answer_text(r["answer"])) for i in ID_RE.findall(g)})

def id_to_sec(cid):
    c = retriever.get(cid)
    return (c["act_code"], c["section"]) if c else None

done = {}
if RUNS_FILE.exists():
    for line in open(RUNS_FILE, encoding="utf-8"):
        rec = json.loads(line); done[rec["id"]] = rec
print(f"{len(done)} cases already done")

for case in TESTSET:
    if case["id"] in done:
        continue
    n0, t0 = len(USAGE_LOG), time.time()
    try:
        r = ask(case["text"])
    except Exception as e:
        print(f"{case['id']} FAILED: {type(e).__name__}: {e} — re-run this cell later to resume"); break
    calls = USAGE_LOG[n0:]
    rec = {"id": case["id"], "input": case["text"], "latency_s": round(time.time() - t0, 1),
           "usd": sum(c["usd"] for c in calls), "tokens": sum(c["in_tok"] + c["out_tok"] for c in calls),
           "n_calls": len(calls), "intake": r["intake"], "answer": r["answer"],
           "sources": [s["chunk_id"] for s in r["sources"]], "cited": cited_ids(r),
           "verification": r["verification"], "support": r["support"] and
               {"ok": r["support"]["ok"], "problems": r["support"]["problems"]},
           "attempts": r["attempts"], "mode": result_mode(r), "fallback": is_fallback(r)}
    with open(RUNS_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    done[case["id"]] = rec
    print(f"{case['id']}  mode={rec['mode']:<17} fallback={rec['fallback']!s:<5} cited={rec['cited'][:3]}  {rec['latency_s']}s")
    time.sleep(SLEEP_BETWEEN)

RUNS = [done[c["id"]] for c in TESTSET if c["id"] in done]
print(f"\n{len(RUNS)}/{len(TESTSET)} cases complete")

18 cases already done
F01  mode=section_not_found fallback=False cited=[]  0.0s
F02  mode=section_not_found fallback=False cited=[]  0.1s
O01  mode=out_of_scope      fallback=False cited=[]  0.6s
O02  mode=out_of_scope      fallback=False cited=[]  0.8s
O03  mode=out_of_scope      fallback=False cited=[]  0.9s
L01  mode=answer            fallback=False cited=['KRA-s49-p1']  6.9s
L02  mode=answer            fallback=True  cited=[]  67.4s
  LLM error (RateLimitError), retrying in 4s
L03  mode=answer            fallback=False cited=['TPA-s108-p1', 'TPA-s108-p2', 'TPA-s108-p3']  47.2s
  LLM error (RateLimitError), retrying in 4s
  LLM error (RateLimitError), retrying in 8s
L04 FAILED: RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01k6qwm28vf31r25m582hy0fq6` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 196852, Requested 3803. Please try again in 4m42.96s. Need more tokens? Upgrade to D

In [69]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 1.0
    r = tp / (tp + fn) if tp + fn else 1.0
    return round(p, 2), round(r, 2), round(2 * p * r / (p + r), 2) if p + r else 0.0

gold_by_id = {c["id"]: c for c in TESTSET}
rows = []
for run in RUNS:
    g, it = gold_by_id[run["id"]], run["intake"]
    rows.append({
        "id": run["id"], "cat": g["cat"],
        "mode_ok": run["mode"] == g["mode"],
        "scope_ok": bool(it.get("in_scope")) == g["in_scope"] or run["mode"] == g["mode"] == "section_not_found",
        "topic_ok": g["topics"] is None or it.get("topic") in g["topics"],
        "lang_ok": it.get("language") == g["lang"],
        "esc_pred": bool(run["answer"].get("needs_lawyer")) and bool(it["escalate"]["needs_lawyer"]),
        "esc_any": bool(run["answer"].get("needs_lawyer")),
        "esc_gold": g["escalate"],
    })
routing = pd.DataFrame(rows)

# escalation measured on the CODE decision (rules), and separately on the final banner (code OR model)
tp = ((routing.esc_pred) & routing.esc_gold).sum(); fp = ((routing.esc_pred) & ~routing.esc_gold).sum()
fn = ((~routing.esc_pred) & routing.esc_gold).sum()
p, r_, f1 = prf(tp, fp, fn)
banner_rate_non_urgent = routing[~routing.esc_gold & (routing.cat != "out_of_scope")].esc_any.mean()

ROUTING_METRICS = {
    "mode accuracy": routing.mode_ok.mean(),
    "scope accuracy": routing.scope_ok.mean(),
    "topic accuracy": routing.topic_ok.mean(),
    "language detection accuracy": routing.lang_ok.mean(),
    "escalation precision (code rules)": p, "escalation recall (code rules)": r_, "escalation F1 (code rules)": f1,
    "lawyer banner shown on non-urgent cases": banner_rate_non_urgent,
    "fake sections handled without LLM": routing[routing.cat == "fake_section"].mode_ok.mean(),
    "out-of-scope correctly refused": routing[routing.cat == "out_of_scope"].mode_ok.mean(),
}
for k, v in ROUTING_METRICS.items():
    print(f"{k:<45} {v:.0%}" if isinstance(v, float) else f"{k:<45} {v}")
print("\nFailures:"); print(routing[~(routing.mode_ok & routing.topic_ok & routing.lang_ok)][["id", "cat", "mode_ok", "topic_ok", "lang_ok"]])

mode accuracy                                 100%
scope accuracy                                100%
topic accuracy                                100%
language detection accuracy                   100%
escalation precision (code rules)             100%
escalation recall (code rules)                100%
escalation F1 (code rules)                    100%
lawyer banner shown on non-urgent cases       84%
fake sections handled without LLM             100%
out-of-scope correctly refused                100%

Failures:
Empty DataFrame
Columns: [id, cat, mode_ok, topic_ok, lang_ok]
Index: []


In [70]:
def gold_match(sec_pair, gold):
    return sec_pair is not None and any(sec_pair[0] == a and (s is None or sec_pair[1] == s) for a, s in gold)

ret_rows = []
for run in RUNS:
    g = gold_by_id[run["id"]]
    if g["mode"] != "answer" or not g["gold"] or not run["sources"]:
        continue
    secs = [id_to_sec(cid) for cid in run["sources"]]
    rank = next((i + 1 for i, s in enumerate(secs) if gold_match(s, g["gold"])), None)
    ret_rows.append({"id": run["id"], "lang": g["lang"], "rank": rank,
                     "hit@1": rank == 1, "hit@5": rank is not None and rank <= 5,
                     "rr": 1 / rank if rank else 0.0})
retrieval = pd.DataFrame(ret_rows)

RETRIEVAL_METRICS = {"cases": len(retrieval), "hit@1": retrieval["hit@1"].mean(),
                     "hit@5": retrieval["hit@5"].mean(), "MRR": retrieval["rr"].mean()}
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in RETRIEVAL_METRICS.items()})
print("\nBy language:"); print(retrieval.groupby("lang")[["hit@1", "hit@5", "rr"]].mean().round(2))
print("\nMisses:"); print(retrieval[~retrieval["hit@5"]])

{'cases': 18, 'hit@1': np.float64(0.667), 'hit@5': np.float64(0.944), 'MRR': np.float64(0.787)}

By language:
      hit@1  hit@5    rr
lang                    
en      0.6   0.93  0.74
hi      1.0   1.00  1.00
kn      1.0   1.00  1.00

Misses:
     id lang  rank  hit@1  hit@5   rr
10  G11   en   NaN  False  False  0.0


In [71]:
SCRIPT_RANGES = {"devanagari": r"[\u0900-\u097F]", "kannada": r"[\u0C80-\u0CFF]", "latin": r"[A-Za-z]"}
EXPECTED_SCRIPT = {"en": "latin", "hinglish": "latin", "hi": "devanagari", "kn": "kannada"}

def dominant_script(text):
    counts = {k: len(re.findall(p, text)) for k, p in SCRIPT_RANGES.items()}
    return max(counts, key=counts.get)

g_rows = []
for run in RUNS:
    g = gold_by_id[run["id"]]
    if run["mode"] != "answer":
        continue
    text = answer_text(run["answer"])
    retrieved_secs = {id_to_sec(c)[1] for c in run["sources"] if id_to_sec(c)}
    mentioned = {s.upper() for s in SEC_MENTION_RE.findall(text)}
    cited_secs = [id_to_sec(c) for c in run["cited"]]
    g_rows.append({
        "id": run["id"], "cat": g["cat"], "fallback": run["fallback"],
        "first_pass": run["attempts"] == 1 and not run["fallback"],
        "hallucinated_sections": sorted(mentioned - retrieved_secs),
        "case_law": bool(CASE_LAW_RE.search(text)),
        "citations_valid": all(c in run["sources"] for c in run["cited"]),
        "cites_gold": (any(gold_match(s, g["gold"]) for s in cited_secs) if g["gold"] else None),
        "must_not_violated": [m for m in g.get("must_not", []) if m.lower() in text.lower()],
        "script_ok": dominant_script(text) == EXPECTED_SCRIPT.get(g["lang"], "latin"),
    })
grounding = pd.DataFrame(g_rows)
non_fb = grounding[~grounding.fallback]

GROUNDING_METRICS = {
    "answers with hallucinated section numbers": (grounding.hallucinated_sections.str.len() > 0).mean(),
    "answers mentioning case law": grounding.case_law.mean(),
    "all citations point to retrieved text": grounding.citations_valid.mean(),
    "must-not violations (e.g. '12%' on deposit)": (grounding.must_not_violated.str.len() > 0).mean(),
    "passed all checks on first attempt": grounding.first_pass.mean(),
    "fell back to 'can't verify'": grounding.fallback.mean(),
    "cites a correct (gold) section, excl. fallbacks": non_fb.cites_gold.dropna().mean(),
    "replied in the user's script": grounding.script_ok.mean(),
}
for k, v in GROUNDING_METRICS.items():
    print(f"{k:<52} {v:.0%}")
print("\nProblem rows:")
print(grounding[(grounding.hallucinated_sections.str.len() > 0) | grounding.case_law |
                (grounding.must_not_violated.str.len() > 0) | ~grounding.script_ok])

answers with hallucinated section numbers            0%
answers mentioning case law                          0%
all citations point to retrieved text                100%
must-not violations (e.g. '12%' on deposit)          0%
passed all checks on first attempt                   24%
fell back to 'can't verify'                          52%
cites a correct (gold) section, excl. fallbacks      100%
replied in the user's script                         95%

Problem rows:
     id           cat  fallback  first_pass hallucinated_sections  case_law  \
19  L02  multilingual      True       False                    []     False   

    citations_valid cites_gold must_not_violated  script_ok  
19             True      False                []      False  


In [72]:
JUDGE_MODEL = next((m for m in ["llama-3.3-70b-versatile", "moonshotai/kimi-k2-instruct", "qwen/qwen3-32b", MODEL_SMALL]
                    if m in available and m != MODEL_BIG), MODEL_SMALL)
print("Judge model:", JUDGE_MODEL, "(answerer:", MODEL_BIG, ")")

JUDGE_SYSTEM = """You evaluate answers from a tenancy-law help tool for Bengaluru. You get the USER MESSAGE,
the LAW SOURCES the tool retrieved, the ANSWER, and the EXPECTED BEHAVIOUR for this test case.
Score 1-5 (5 = best):
- faithfulness: every legal claim is supported by the SOURCES; nothing invented
- helpfulness: gives the user a clear sense of where they stand and concrete next steps
- clarity: plain, simple language a non-lawyer understands
- honesty: admits uncertainty/gaps, recommends a lawyer when appropriate, no false confidence
And "behaviour_pass": true/false — does the answer satisfy the EXPECTED BEHAVIOUR?
Return ONLY JSON: {"faithfulness": int, "helpfulness": int, "clarity": int, "honesty": int,
                   "behaviour_pass": bool, "comment": str}"""

JUDGE_FILE = OUT_DIR / "eval_judge.jsonl"
judged = {}
if JUDGE_FILE.exists():
    for line in open(JUDGE_FILE, encoding="utf-8"):
        rec = json.loads(line); judged[rec["id"]] = rec

for run in RUNS:
    if run["id"] in judged:
        continue
    g = gold_by_id[run["id"]]
    srcs = [retriever.get(c) for c in run["sources"]]
    payload = (f"USER MESSAGE:\n{run['input']}\n\nLAW SOURCES:\n{format_sources([s for s in srcs if s]) or '(none — routed without retrieval)'}\n\n"
               f"ANSWER:\n{json.dumps(run['answer'], ensure_ascii=False)}\n\nEXPECTED BEHAVIOUR:\n{g['expect']}")
    try:
        j = llm([{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": payload}],
                JUDGE_MODEL, json_mode=True, temperature=0, tag="judge")
    except Exception as e:
        print(f"{run['id']} judge failed ({type(e).__name__}) — re-run later"); break
    j["id"] = run["id"]
    with open(JUDGE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")
    judged[run["id"]] = j
    time.sleep(SLEEP_BETWEEN)

judge = pd.DataFrame([judged[r["id"]] for r in RUNS if r["id"] in judged])
judge["cat"] = judge["id"].map(lambda i: gold_by_id[i]["cat"])
JUDGE_METRICS = {k: judge[k].mean() for k in ["faithfulness", "helpfulness", "clarity", "honesty"]}
JUDGE_METRICS["behaviour pass rate"] = judge["behaviour_pass"].mean()
print({k: round(v, 2) for k, v in JUDGE_METRICS.items()})
print("\nBy category:"); print(judge.groupby("cat")[["faithfulness", "helpfulness", "honesty", "behaviour_pass"]].mean().round(2))
print("\nBehaviour failures:"); print(judge[~judge.behaviour_pass][["id", "comment"]].to_string())

Judge model: openai/gpt-oss-20b (answerer: openai/gpt-oss-120b )
  JSON mode rejected, retrying as plain text
{'faithfulness': np.float64(3.69), 'helpfulness': np.float64(3.38), 'clarity': np.float64(3.85), 'honesty': np.float64(4.54), 'behaviour pass rate': np.float64(0.54)}

By category:
              faithfulness  helpfulness  honesty  behaviour_pass
cat                                                             
escalation            3.75         3.00     4.50            0.50
fake_section          5.00         4.50     5.00            1.00
gap_or_myth           4.00         4.67     4.00            0.67
grounded              2.91         2.45     4.45            0.27
multilingual          4.00         4.00     4.67            0.67
out_of_scope          5.00         4.67     5.00            1.00

Behaviour failures:
     id                                                                                                                                                                 

In [58]:
CONSIST_IDS = ["G01", "G05", "G07", "M01", "E01"]
N_REPEATS = 2                      # plus the original run = 3 runs each

def jaccard(a, b):
    a, b = set(a), set(b)
    return 1.0 if not a and not b else len(a & b) / len(a | b)

cons_rows = []
for cid in CONSIST_IDS:
    base = done[cid]
    runs = [base]
    for _ in range(N_REPEATS):
        r = ask(gold_by_id[cid]["text"])
        runs.append({"mode": result_mode(r), "fallback": is_fallback(r),
                     "cited": cited_ids(r), "topic": r["intake"].get("topic")})
        time.sleep(SLEEP_BETWEEN)
    sec_sets = [{id_to_sec(c)[1] for c in x["cited"] if id_to_sec(c)} for x in runs]
    pairs = [(i, j) for i in range(len(runs)) for j in range(i + 1, len(runs))]
    cons_rows.append({
        "id": cid,
        "topic_agree": len({x.get("topic") or x["intake"].get("topic") for x in runs}) == 1,
        "mode_agree": len({(x["mode"], x["fallback"]) for x in runs}) == 1,
        "cited_section_jaccard": round(sum(jaccard(sec_sets[i], sec_sets[j]) for i, j in pairs) / len(pairs), 2),
        "fallbacks": sum(x["fallback"] for x in runs),
    })
consistency = pd.DataFrame(cons_rows)
CONSISTENCY_METRICS = {"topic agreement": consistency.topic_agree.mean(),
                       "mode agreement (incl. fallback)": consistency.mode_agree.mean(),
                       "mean cited-section Jaccard": consistency.cited_section_jaccard.mean()}
print(consistency); print({k: round(v, 2) for k, v in CONSISTENCY_METRICS.items()})

    id  topic_agree  mode_agree  cited_section_jaccard  fallbacks
0  G01         True        True                   1.00          3
1  G05         True        True                   1.00          3
2  G07        False       False                   0.33          2
3  M01         True       False                   0.00          1
4  E01         True        True                   1.00          3
{'topic agreement': np.float64(0.8), 'mode agreement (incl. fallback)': np.float64(0.6), 'mean cited-section Jaccard': np.float64(0.67)}


In [73]:
ops = pd.DataFrame(RUNS)
answered = ops[ops["mode"] == "answer"]
OPS_METRICS = {
    "latency p50 (s), all": ops.latency_s.median(),
    "latency p95 (s), all": ops.latency_s.quantile(0.95),
    "latency p50 (s), full answers": answered.latency_s.median(),
    "avg LLM calls / conversation": ops.n_calls.mean(),
    "avg tokens / conversation": ops.tokens.mean(),
    "avg cost / conversation (USD, paid rates)": ops.usd.mean(),
}
for k, v in OPS_METRICS.items():
    print(f"{k:<45} {v:,.4f}" if "USD" in k else f"{k:<45} {v:,.1f}")

per = ops.usd.mean()
print("\nScale projection (LLM cost only):")
for n in [1_000, 100_000, 1_000_000]:
    print(f"  {n:>9,} conversations/month ≈ ${per * n:,.0f}  (≈ ₹{per * n * 84:,.0f})")
print("Free-tier note: Groq free limits cap daily volume; paid tier needed beyond prototype scale.")

latency p50 (s), all                          44.2
latency p95 (s), all                          90.4
latency p50 (s), full answers                 53.3
avg LLM calls / conversation                  3.3
avg tokens / conversation                     7,453.2
avg cost / conversation (USD, paid rates)     0.0020

Scale projection (LLM cost only):
      1,000 conversations/month ≈ $2  (≈ ₹172)
    100,000 conversations/month ≈ $204  (≈ ₹17,154)
  1,000,000 conversations/month ≈ $2,042  (≈ ₹171,538)
Free-tier note: Groq free limits cap daily volume; paid tier needed beyond prototype scale.


In [60]:
AUDIT_FILE = OUT_DIR / "human_audit.csv"
if not AUDIT_FILE.exists():
    audit_rows = []
    for run in RUNS:
        if run["mode"] != "answer":
            continue
        audit_rows.append({"id": run["id"], "question": run["input"],
                           "answer": run["answer"]["where_you_stand"][:600],
                           "cited": ", ".join(run["cited"]),
                           "judge_behaviour_pass": judged.get(run["id"], {}).get("behaviour_pass"),
                           "human_verdict": "",      # fill: correct / partly / wrong
                           "human_note": ""})
    pd.DataFrame(audit_rows).to_csv(AUDIT_FILE, index=False)
    print(f"Created {AUDIT_FILE} — open it, read each answer against the cited law text "
          "(use show_sections), and fill human_verdict with: correct / partly / wrong. Then run Cell 31.")
else:
    print("Audit file exists:", AUDIT_FILE)

Created /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/human_audit.csv — open it, read each answer against the cited law text (use show_sections), and fill human_verdict with: correct / partly / wrong. Then run Cell 31.


In [77]:
audit = pd.read_csv(AUDIT_FILE, dtype=str, keep_default_na=False)      # read everything as text
audit["human_verdict"] = audit["human_verdict"].str.strip().str.lower()
audit = audit[audit.human_verdict.isin(["correct", "partly", "wrong"])]

if len(audit):
    audit["human_pass"] = audit.human_verdict.eq("correct")
    audit["judge_pass"] = audit.judge_behaviour_pass.str.strip().str.lower().eq("true")
    po = (audit.human_pass == audit.judge_pass).mean()
    ph, pj = audit.human_pass.mean(), audit.judge_pass.mean()
    pe = ph * pj + (1 - ph) * (1 - pj)
    kappa = (po - pe) / (1 - pe) if pe < 1 else 1.0
    HUMAN_METRICS = {"audited answers": len(audit),
                     "human: correct": audit.human_verdict.eq("correct").mean(),
                     "human: partly": audit.human_verdict.eq("partly").mean(),
                     "human: wrong": audit.human_verdict.eq("wrong").mean(),
                     "judge–human agreement": po, "Cohen's kappa": kappa}
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in HUMAN_METRICS.items()})
    print("\nDisagreements:")
    print(audit[audit.human_pass != audit.judge_pass][["id", "human_verdict", "judge_behaviour_pass", "human_note"]])
else:
    HUMAN_METRICS = {}
    print(f"No graded rows yet. Open {AUDIT_FILE}, fill the 'human_verdict' column with "
          "correct / partly / wrong, save, then re-run this cell.")

No graded rows yet. Open /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/human_audit_v2.csv, fill the 'human_verdict' column with correct / partly / wrong, save, then re-run this cell.


In [76]:
#Updated Cell 30 

AUDIT_FILE = OUT_DIR / "human_audit_v2.csv"

def cited_law_excerpt(cids, max_chars=500):
    """The actual law text the answer cites, so the grader can check it in the sheet."""
    parts = []
    for cid in cids:
        c = retriever.get(cid)
        if c:
            label = c["section"] if c["section"].startswith("Schedule") else f"s.{c['section']}"
            parts.append(f"[{cid}] {c['act_code']} {label} — {c['title']}: {c['text'][:max_chars]}")
    return "\n\n".join(parts)

if AUDIT_FILE.exists():
    print("Audit file already exists — not overwriting:", AUDIT_FILE)
else:
    audit_rows = []
    for run in RUNS:
        if run["mode"] != "answer":                      # out-of-scope / fake-section are code replies, nothing to grade
            continue
        a = run["answer"]
        audit_rows.append({
            "id": run["id"],
            "category": gold_by_id[run["id"]]["cat"],
            "language": gold_by_id[run["id"]]["lang"],
            "question": run["input"],
            "fallback": run["fallback"],
            "answer": a.get("where_you_stand", ""),
            "options": " | ".join(a.get("options", [])),
            "next_steps": " | ".join(a.get("next_steps", [])),
            "cited": ", ".join(run["cited"]),
            "cited_law_text": cited_law_excerpt(run["cited"]),
            "expected_behaviour": gold_by_id[run["id"]]["expect"],
            "judge_behaviour_pass": judged.get(run["id"], {}).get("behaviour_pass"),
            "human_verdict": "",        # fill: correct / partly / wrong
            "human_note": "",
        })
    pd.DataFrame(audit_rows).to_csv(AUDIT_FILE, index=False, encoding="utf-8-sig")   # utf-8-sig: Hindi/Kannada show correctly in Excel
    print(f"Created {AUDIT_FILE.name} with {len(audit_rows)} rows "
          f"({sum(r['fallback'] for r in audit_rows)} are fallbacks).")
    print("Fill 'human_verdict' with: correct / partly / wrong → save as CSV → run Cell 31.")

Audit file already exists — not overwriting: /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs/human_audit_v2.csv


In [78]:
def pct(v): return f"{v:.0%}" if isinstance(v, float) and v <= 1 else (f"{v:.2f}" if isinstance(v, float) else str(v))

scorecard = []
for layer, metrics in [("Routing & intake", ROUTING_METRICS), ("Retrieval", RETRIEVAL_METRICS),
                       ("Grounding & language", GROUNDING_METRICS), ("Answer quality (LLM judge, 1–5)", JUDGE_METRICS),
                       ("Consistency", CONSISTENCY_METRICS), ("Operations", OPS_METRICS), ("Human audit", HUMAN_METRICS)]:
    for k, v in metrics.items():
        scorecard.append({"layer": layer, "metric": k,
                          "value": f"{v:.2f}" if layer.startswith("Answer quality") and k != "behaviour pass rate"
                                   else (f"{v:,.4f}" if "USD" in k else (f"{v:.1f}" if "latency" in k or "avg" in k else pct(v)))})
scorecard = pd.DataFrame(scorecard)
scorecard.to_csv(OUT_DIR / "scorecard.csv", index=False)
routing.to_csv(OUT_DIR / "eval_routing.csv", index=False)
retrieval.to_csv(OUT_DIR / "eval_retrieval.csv", index=False)
grounding.to_csv(OUT_DIR / "eval_grounding.csv", index=False)
judge.to_csv(OUT_DIR / "eval_judge.csv", index=False)
consistency.to_csv(OUT_DIR / "eval_consistency.csv", index=False)
print("Saved all eval tables to", OUT_DIR)
pd.set_option("display.max_rows", 100)
scorecard

Saved all eval tables to /Users/tusharsarda06/Kuku FM Project/ai_lawyer/tests/outputs


,layer,metric,value
0,Routing & intake,mode accuracy,100%
1,Routing & intake,scope accuracy,100%
2,Routing & intake,topic accuracy,100%
3,Routing & intake,language detection accuracy,100%
4,Routing & intake,escalation precision (code rules),100%
5,Routing & intake,escalation recall (code rules),100%
6,Routing & intake,escalation F1 (code rules),100%
7,Routing & intake,lawyer banner shown on non-urgent cases,84%
8,Routing & intake,fake sections handled without LLM,100%
9,Routing & intake,out-of-scope correctly refused,100%


In [79]:
grounding[~grounding.script_ok][["id", "cat", "fallback"]]

,id,cat,fallback
19,L02,multilingual,True


In [80]:
from pathlib import Path
import faiss, numpy as np

INDEX_DIR = Path("/Users/tusharsarda06/Kuku FM Project/ai_lawyer/data/index")

idx = faiss.read_index(str(INDEX_DIR / "dense.faiss"))
emb = idx.reconstruct_n(0, idx.ntotal)
np.save(INDEX_DIR / "embeddings.npy", emb)

print(emb.shape, "→ saved", INDEX_DIR / "embeddings.npy")

(458, 768) → saved /Users/tusharsarda06/Kuku FM Project/ai_lawyer/data/index/embeddings.npy
